# Phase 2.2 — Corrected Reasoning Attention Calibration

**Three targeted fixes over Phase 2.1:**
1. **Cross-entropy supervised loss** — direct task supervision on the correct MC letter
2. **Layer-wise cosine distillation** — align hidden states across ALL 24 transformer layers
3. **Cosine distance** instead of Euclidean MSE — direction-preserving alignment

**Training logistics:**
- Print every epoch (CE + Cos breakdown)
- Early stopping (patience=15, min_delta=0.0005)
- Checkpoint saved on every improvement
- Resume training from checkpoint automatically

## Step 1 — Install Dependencies

In [1]:
!pip install transformers torch -q

## Step 2 — Setup: Load Models (conditional to avoid OOM)

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def load_model():
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",
    ).to(device)

if 'teacher_model' not in globals():
    teacher_model = load_model()
    for p in teacher_model.parameters():
        p.requires_grad = False
    teacher_model.eval()
    print("Teacher model loaded and frozen.")
else:
    print("Teacher model already in memory — skipping reload.")

if 'student_model' not in globals():
    student_model = load_model()
    for p in student_model.parameters():
        p.requires_grad = False
    student_model.eval()
    print("Student model loaded and frozen.")
else:
    print("Student model already in memory — skipping reload.")

n_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Parameters per model : {n_params:,}")
print("Both models ready.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Teacher model loaded and frozen.


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Student model loaded and frozen.
Parameters per model : 1,711,376,384
Both models ready.


## Step 3 — MC Token IDs and Predict Helper

In [3]:
A_ID = tokenizer.encode("A", add_special_tokens=False)[0]
B_ID = tokenizer.encode("B", add_special_tokens=False)[0]
C_ID = tokenizer.encode("C", add_special_tokens=False)[0]
D_ID = tokenizer.encode("D", add_special_tokens=False)[0]
MC_IDS     = [A_ID, B_ID, C_ID, D_ID]
MC_LETTERS = ["A",  "B",  "C",  "D"]

print(f"Token IDs  A={A_ID}  B={B_ID}  C={C_ID}  D={D_ID}")


def format_prompt(text: str) -> str:
    """Apply SmolLM2 chat template and append 'Answer:' so the model
    predicts the MC letter as the next token."""
    messages = [{"role": "user", "content": text}]
    return (
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        + "Answer:"
    )


def predict_mc(model, prompt_text: str):
    """Single forward pass → (predicted_letter, probability)."""
    formatted = format_prompt(prompt_text)
    inputs    = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs  = F.softmax(logits, dim=-1)
    mc_probs = [probs[0, tid].item() for tid in MC_IDS]
    best_idx = mc_probs.index(max(mc_probs))
    return MC_LETTERS[best_idx], mc_probs[best_idx]


print("predict_mc() defined.")

Token IDs  A=49  B=50  C=51  D=52
predict_mc() defined.


## Step 4 — Golden Tasks Data and Train/Test Split

38 tasks for training the adapter, 8 held out for evaluation.
Split is fixed with `random.seed(42)`.

In [4]:
import json
import random

raw_golden_json = r"""[
  {
    "id": 2,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 5,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 6,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 14,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 15,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 21,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "answer": "D"
  },
  {
    "id": 24,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 30,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "answer": "C"
  },
  {
    "id": 34,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 36,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "answer": "C"
  },
  {
    "id": 46,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 52,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "answer": "D"
  },
  {
    "id": 55,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 63,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 65,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 74,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "answer": "C"
  },
  {
    "id": 86,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 88,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "answer": "B"
  },
  {
    "id": 92,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 95,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "answer": "C"
  },
  {
    "id": 98,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 106,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 116,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "answer": "C"
  },
  {
    "id": 121,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "answer": "B"
  },
  {
    "id": 123,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "answer": "C"
  },
  {
    "id": 124,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 125,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 130,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 135,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 138,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 140,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 141,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Charlie\nB) Eva\nC) Kate\nD) Mia",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Alice\nB) Eva\nC) Kate\nD) Mia",
    "answer": "C"
  },
  {
    "id": 145,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 146,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "answer": "C"
  },
  {
    "id": 153,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 155,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 164,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "answer": "C"
  },
  {
    "id": 169,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "answer": "C"
  },
  {
    "id": 171,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "answer": "C"
  },
  {
    "id": 172,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 173,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 174,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 178,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "answer": "B"
  },
  {
    "id": 180,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "answer": "C"
  },
  {
    "id": 194,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 199,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  }
]"""
golden_tasks = json.loads(raw_golden_json)

random.seed(42)
shuffled = golden_tasks.copy()
random.shuffle(shuffled)

TRAIN_TASKS = shuffled[:38]
TEST_TASKS  = shuffled[38:]

print(f"Golden tasks : {len(golden_tasks)}")
print(f"Train        : {len(TRAIN_TASKS)}")
print(f"Test         : {len(TEST_TASKS)}")
print()
print("Test task IDs and types:")
for t in TEST_TASKS:
    print(f"  ID {t['id']:3d} | {t['task']:<30} | Answer: {t['answer']}")

Golden tasks : 46
Train        : 38
Test         : 8

Test task IDs and types:
  ID  24 | dyck_language                  | Answer: A
  ID  34 | boolean_expressions            | Answer: A
  ID  65 | boolean_expressions            | Answer: A
  ID  74 | navigate                       | Answer: C
  ID  88 | navigate                       | Answer: B
  ID   5 | dyck_language                  | Answer: A
  ID  30 | navigate                       | Answer: C
  ID 173 | boolean_expressions            | Answer: A


## Step 5 — Baseline: Student Predictions BEFORE Adapter Injection

Record pure 0-shot student predictions while the model is completely unmodified.

In [5]:
print("Capturing baseline predictions (unmodified student, 0-shot) ...")
baseline_preds = {}

for t in TEST_TASKS:
    pred, prob = predict_mc(student_model, t["student_prompt"])
    correct    = (pred == t["answer"])
    baseline_preds[t["id"]] = {"pred": pred, "prob": prob, "correct": correct}
    mark = "✅" if correct else "❌"
    print(f"  Task {t['id']:3d} | {t['task']:<30} | True: {t['answer']} | "
          f"Pred: {pred} ({prob:.2f}) {mark}")

n_correct = sum(1 for v in baseline_preds.values() if v["correct"])
print(f"\nBaseline accuracy: {n_correct}/{len(TEST_TASKS)} = {n_correct/len(TEST_TASKS):.0%}")

Capturing baseline predictions (unmodified student, 0-shot) ...
  Task  24 | dyck_language                  | True: A | Pred: B (0.00) ❌
  Task  34 | boolean_expressions            | True: A | Pred: B (0.00) ❌
  Task  65 | boolean_expressions            | True: A | Pred: B (0.00) ❌
  Task  74 | navigate                       | True: C | Pred: B (0.00) ❌
  Task  88 | navigate                       | True: B | Pred: C (0.00) ❌
  Task   5 | dyck_language                  | True: A | Pred: B (0.00) ❌
  Task  30 | navigate                       | True: C | Pred: B (0.00) ❌
  Task 173 | boolean_expressions            | True: A | Pred: B (0.00) ❌

Baseline accuracy: 0/8 = 0%


## Step 6 — RoPE Utilities

In [6]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    if cos.dim() == 3:
        cos = cos.unsqueeze(1)
        sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


print("RoPE utilities defined.")

RoPE utilities defined.


## Step 7 — CalibratedAttention Adapter + Injection

Same architecture as Phase 2.1 — unchanged. Rank-4 `U_q`/`U_k` at every layer.

```
delta  = (Q_rot @ U_q) @ (K_rot @ U_k)^T
S'     = S_frozen + delta
output = softmax(S') @ V
```

In [7]:
class CalibratedAttention(nn.Module):
    """
    Wraps a frozen LlamaAttention and injects:
        S' = S + delta,   delta = (Q_rot @ U_q) @ (K_rot @ U_k)^T
    Only U_q and U_k (float32) are trainable.
    """
    def __init__(self, original_attn, rank: int = 4):
        super().__init__()
        self.original_attn    = original_attn
        self.config           = getattr(original_attn, "config", None)

        self.num_heads        = getattr(original_attn, "num_heads", None) or self.config.num_attention_heads
        self.num_kv_heads     = getattr(original_attn, "num_key_value_heads", None) or self.config.num_key_value_heads
        self.hidden_size      = getattr(original_attn, "hidden_size", None) or self.config.hidden_size
        self.head_dim         = getattr(original_attn, "head_dim", None) or (self.hidden_size // self.num_heads)
        self.num_kv_groups    = self.num_heads // self.num_kv_heads

        self.expected_tuple_len = None

        self.U_q = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        self.U_k = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        nn.init.normal_(self.U_q, std=0.02)
        nn.init.normal_(self.U_k, std=0.02)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        **kwargs,
    ):
        if self.expected_tuple_len is None:
            with torch.no_grad():
                dummy_out = self.original_attn(
                    hidden_states=hidden_states,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_value,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                    cache_position=cache_position,
                    position_embeddings=position_embeddings,
                    **kwargs
                )
                self.expected_tuple_len = len(dummy_out) if isinstance(dummy_out, tuple) else 1

        B, T, _ = hidden_states.shape

        with torch.no_grad():
            Q = self.original_attn.q_proj(hidden_states)
            K = self.original_attn.k_proj(hidden_states)
            V = self.original_attn.v_proj(hidden_states)

        Q = Q.view(B, T, self.num_heads,    self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        with torch.no_grad():
            if position_embeddings is not None:
                cos, sin = position_embeddings
            else:
                cos, sin = self.original_attn.rotary_emb(V, position_ids)
            Q, K = apply_rotary_pos_emb(Q, K, cos.float(), sin.float())

            if self.num_kv_groups > 1:
                K = K.repeat_interleave(self.num_kv_groups, dim=1)
                V = V.repeat_interleave(self.num_kv_groups, dim=1)

        Q = Q.detach().float()
        K = K.detach().float()
        V = V.detach().float()

        A     = torch.einsum('bhtd,hdr->bhtr', Q, self.U_q)
        Bm    = torch.einsum('bhtd,hdr->bhtr', K, self.U_k)
        delta = torch.matmul(A, Bm.transpose(-1, -2))

        with torch.no_grad():
            S      = torch.matmul(Q, K.transpose(-1, -2)) / (self.head_dim ** 0.5)
            causal = torch.tril(torch.ones(T, T, device=device)).bool()
            S      = S.masked_fill(~causal, torch.finfo(S.dtype).min)

        S_prime = S + delta
        if attention_mask is not None:
            S_prime = S_prime + attention_mask.float()
        attn_weights = F.softmax(S_prime, dim=-1)

        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(B, T, self.num_heads * self.head_dim)
        attn_output = attn_output.to(dtype=hidden_states.dtype)
        attn_output = self.original_attn.o_proj(attn_output)

        return (attn_output, None, None, None)[:self.expected_tuple_len]


def inject_calibration(model, rank=4):
    trainable_params = []
    for layer in model.model.layers:
        cal = CalibratedAttention(layer.self_attn, rank=rank).to(device)
        layer.self_attn = cal
        trainable_params.extend([cal.U_q, cal.U_k])
    return trainable_params


if 'trainable_params' not in globals():
    trainable_params = inject_calibration(student_model, rank=4)
    print("Injection complete.")
else:
    print("Adapter already injected. Restart kernel to reinject fresh.")

n_trainable = sum(p.numel() for p in trainable_params)
print(f"Trainable scalars : {n_trainable:,}  (U_q + U_k across all layers)")

Injection complete.
Trainable scalars : 393,216  (U_q + U_k across all layers)


## Step 8 — Gradient Check

In [8]:
chk_task   = TRAIN_TASKS[0]
inputs_chk = tokenizer(format_prompt(chk_task["student_prompt"]),
                       return_tensors="pt").to(device)
out_chk    = student_model(**inputs_chk, output_hidden_states=True)
hs_chk     = out_chk.hidden_states[-1][:, -1, :]

assert hs_chk.requires_grad, "❌ Gradient check FAILED — U_q/U_k not in graph!"
print("✅ Gradient check passed — U_q and U_k are in the computation graph.")

✅ Gradient check passed — U_q and U_k are in the computation graph.


## Step 9 — Pre-compute Teacher Targets (ALL Layers)

**Key change from Phase 2.1:** We now store the teacher's hidden state at
**every transformer layer** (not just the last one). This gives the distillation
loss a rich, multi-scale signal across the entire network.

Teacher runs once per training task. Never runs again after this cell.

In [9]:
print(f"Pre-computing teacher targets for {len(TRAIN_TASKS)} training tasks ...")
print("Storing hidden states at ALL layers per task (one-time, cached)\n")

train_pairs = []

for i, t in enumerate(TRAIN_TASKS):
    teacher_inputs = tokenizer(format_prompt(t["teacher_prompt"]),
                               return_tensors="pt").to(device)
    student_inputs = tokenizer(format_prompt(t["student_prompt"]),
                               return_tensors="pt").to(device)

    with torch.no_grad():
        out_t = teacher_model(**teacher_inputs, output_hidden_states=True)

        # All hidden states: shape list of (1, seq_len, hidden_dim)
        # We take the last token at each layer → list of (1, hidden_dim) float32
        t_hs_layers = [
            out_t.hidden_states[l][:, -1, :].float().detach()
            for l in range(len(out_t.hidden_states))
        ]

    # correct_idx: position in [A,B,C,D] space (0=A, 1=B, 2=C, 3=D)
    correct_idx = torch.tensor(
        [MC_LETTERS.index(t["answer"])], dtype=torch.long, device=device
    )

    train_pairs.append((student_inputs, t_hs_layers, correct_idx))

    if (i + 1) % 10 == 0 or i == 0:
        n_layers = len(t_hs_layers)
        print(f"  [{i+1:02d}/{len(TRAIN_TASKS)}] Task {t['id']:3d} | {t['task']}"
              f"  ({n_layers} hidden-state layers stored)")

n_layers = len(train_pairs[0][1])
print(f"\nDone. {len(train_pairs)} training pairs cached.")
print(f"Hidden state layers per task : {n_layers}  (embedding + {n_layers-1} transformer layers)")

Pre-computing teacher targets for 38 training tasks ...
Storing hidden states at ALL layers per task (one-time, cached)

  [01/38] Task  92 | dyck_language  (25 hidden-state layers stored)
  [10/38] Task  14 | dyck_language  (25 hidden-state layers stored)
  [20/38] Task 140 | boolean_expressions  (25 hidden-state layers stored)
  [30/38] Task  86 | multistep_arithmetic  (25 hidden-state layers stored)

Done. 38 training pairs cached.
Hidden state layers per task : 25  (embedding + 24 transformer layers)


## Step 10 — Test Results BEFORE Training

Adapter injected, weights are random (not yet trained).

In [10]:
student_model.eval()
print("\n" + "="*72)
print("TEST SET — BEFORE TRAINING  (adapter injected, U_q/U_k not yet trained)")
print("="*72)

for t in TEST_TASKS:
    pred, prob = predict_mc(student_model, t["student_prompt"])
    correct    = (pred == t["answer"])
    base       = baseline_preds[t["id"]]
    mark       = "✅" if correct else "❌"
    base_mark  = "✅" if base["correct"] else "❌"
    print(f"Task {t['id']:3d} | {t['task']:<28} | True: {t['answer']} | "
          f"Baseline: {base['pred']} {base_mark} | Pre-train: {pred} ({prob:.2f}) {mark}")


TEST SET — BEFORE TRAINING  (adapter injected, U_q/U_k not yet trained)
Task  24 | dyck_language                | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  34 | boolean_expressions          | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  65 | boolean_expressions          | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  74 | navigate                     | True: C | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  88 | navigate                     | True: B | Baseline: C ❌ | Pre-train: B (0.00) ✅
Task   5 | dyck_language                | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  30 | navigate                     | True: C | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task 173 | boolean_expressions          | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌


## Step 11 — Training

**Loss:** `1.0 × CE  +  0.1 × LayerCosine(all layers)  +  0.001 × L2`

- **CE**: direct cross-entropy on the correct MC letter (new — the most important fix)
- **LayerCosine**: average `1 - cosine_similarity` across all transformer layers
- **L2**: adapter weight regularization

**Training logistics:**
- Prints every epoch with CE and Cosine breakdown
- Saves checkpoint whenever loss improves
- Early stopping when loss doesn't improve by `MIN_DELTA` for `PATIENCE` epochs
- **Resume**: re-run this cell after interruption — it automatically loads the checkpoint

In [11]:
EPOCHS         = 100
LR             = 1e-3
PATIENCE       = 15
MIN_DELTA      = 0.0005
CHECKPOINT     = 'phase22_adapter_checkpoint.pt'

# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer = optim.AdamW(trainable_params, lr=LR, weight_decay=0.0)

# ── Resume from checkpoint if it exists ───────────────────────────────────────
start_epoch     = 1
best_loss       = float('inf')
epochs_no_impr  = 0

if os.path.exists(CHECKPOINT):
    print(f"Found checkpoint: {CHECKPOINT}")
    ckpt = torch.load(CHECKPOINT, map_location=device)
    for p, saved in zip(trainable_params, ckpt['adapter_weights']):
        p.data.copy_(saved.to(device))
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch    = ckpt['epoch'] + 1
    best_loss      = ckpt['best_loss']
    epochs_no_impr = ckpt['epochs_no_impr']
    print(f"Resumed from epoch {ckpt['epoch']:03d} | "
          f"Best loss: {best_loss:.4f} | "
          f"Patience counter: {epochs_no_impr}/{PATIENCE}")
else:
    print("No checkpoint found — starting fresh training.")

print(f"\nTraining epochs {start_epoch}→{EPOCHS}  |  lr={LR}")
print(f"Loss = 1.0×CE  +  0.1×LayerCosine({n_layers} layers)  +  0.001×L2")
print(f"Early stop: patience={PATIENCE}, min_delta={MIN_DELTA}\n")

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    student_model.train()
    optimizer.zero_grad()

    total_loss = 0.0
    total_ce   = 0.0
    total_cos  = 0.0

    for s_inputs, t_hs_layers, correct_idx in train_pairs:
        out = student_model(**s_inputs, output_hidden_states=True)

        # ── CE loss: direct task supervision ──────────────────────────────
        full_logits = out.logits[:, -1, :].float()          # (1, vocab)
        mc_logits   = full_logits[:, MC_IDS]                # (1, 4)
        ce = F.cross_entropy(mc_logits, correct_idx)

        # ── Layer-wise cosine distillation ────────────────────────────────
        # Sum 1 - cos_sim across ALL layers, then average
        cos_loss = sum(
            1.0 - F.cosine_similarity(
                out.hidden_states[l][:, -1, :].float(),
                t_hs_layers[l],
                dim=-1
            ).mean()
            for l in range(n_layers)
        ) / n_layers

        # ── L2 regularization on adapter weights ──────────────────────────
        l2 = sum(p.pow(2).sum() for p in trainable_params)

        loss = 1.0 * ce + 0.1 * cos_loss + 0.001 * l2
        loss.backward()

        total_loss += loss.item()
        total_ce   += ce.item()
        total_cos  += cos_loss.item()

    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    avg_loss = total_loss / len(train_pairs)
    avg_ce   = total_ce   / len(train_pairs)
    avg_cos  = total_cos  / len(train_pairs)

    # ── Early stopping + checkpoint ───────────────────────────────────────
    improved = avg_loss < best_loss - MIN_DELTA
    if improved:
        best_loss      = avg_loss
        epochs_no_impr = 0
        torch.save({
            'epoch':           epoch,
            'best_loss':       best_loss,
            'epochs_no_impr':  epochs_no_impr,
            'adapter_weights': [p.data.cpu() for p in trainable_params],
            'optimizer_state': optimizer.state_dict(),
        }, CHECKPOINT)

    suffix = " ← best ✓ (saved)" if improved else f"  [{epochs_no_impr}/{PATIENCE}]"
    print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f}  CE: {avg_ce:.4f}  Cos: {avg_cos:.4f}{suffix}")

    if not improved:
        epochs_no_impr += 1
        if epochs_no_impr >= PATIENCE:
            print(f"\nEarly stopping — no improvement for {PATIENCE} consecutive epochs.")
            break

print(f"\nTraining complete. Best loss: {best_loss:.4f}")

No checkpoint found — starting fresh training.

Training epochs 1→100  |  lr=0.001
Loss = 1.0×CE  +  0.1×LayerCosine(25 layers)  +  0.001×L2
Early stop: patience=15, min_delta=0.0005

Epoch 001 | Loss: 2.1295  CE: 1.9682  Cos: 0.0456 ← best ✓ (saved)
Epoch 002 | Loss: 1.5573  CE: 1.3982  Cos: 0.0419 ← best ✓ (saved)
Epoch 003 | Loss: 1.2207  CE: 1.0629  Cos: 0.0418 ← best ✓ (saved)
Epoch 004 | Loss: 1.0108  CE: 0.8537  Cos: 0.0447 ← best ✓ (saved)
Epoch 005 | Loss: 0.8620  CE: 0.7052  Cos: 0.0495 ← best ✓ (saved)
Epoch 006 | Loss: 0.7350  CE: 0.5784  Cos: 0.0558 ← best ✓ (saved)
Epoch 007 | Loss: 0.6418  CE: 0.4850  Cos: 0.0636 ← best ✓ (saved)
Epoch 008 | Loss: 0.5768  CE: 0.4201  Cos: 0.0730 ← best ✓ (saved)
Epoch 009 | Loss: 0.5320  CE: 0.3754  Cos: 0.0833 ← best ✓ (saved)
Epoch 010 | Loss: 0.5017  CE: 0.3452  Cos: 0.0935 ← best ✓ (saved)
Epoch 011 | Loss: 0.4730  CE: 0.3169  Cos: 0.1038 ← best ✓ (saved)
Epoch 012 | Loss: 0.4443  CE: 0.2884  Cos: 0.1138 ← best ✓ (saved)
Epoch 013 | 

## Step 12 — Test Results AFTER Training

Three-way comparison: **Baseline** (pure 0-shot) | **Before training** | **After training**

Any ❌→✅ flip = adapter successfully recalibrated attention for that task.

In [12]:
student_model.eval()

print("\n" + "="*85)
print("TEST SET — AFTER TRAINING")
print("="*85)
print(f"{'ID':<6} {'Type':<30} {'True':>5} {'Baseline':>10} {'After':>8}  {'Change'}")
print("="*85)

after_correct = 0
flipped       = 0
regressed     = 0

for t in TEST_TASKS:
    base       = baseline_preds[t["id"]]
    pred, prob = predict_mc(student_model, t["student_prompt"])
    correct    = (pred == t["answer"])

    if correct:
        after_correct += 1

    b_mark = "✅" if base["correct"] else "❌"
    a_mark = "✅" if correct else "❌"

    if not base["correct"] and correct:
        change = "← RECALIBRATED ✨"
        flipped += 1
    elif base["correct"] and not correct:
        change = "← REGRESSED ⚠️"
        regressed += 1
    else:
        change = ""

    print(f"{t['id']:<6} {t['task']:<30} {t['answer']:>5} "
          f"  {base['pred']} {b_mark}      {pred} {a_mark}  {change}")

print("="*85)
before_n = sum(1 for v in baseline_preds.values() if v["correct"])
print(f"\nBaseline (pure 0-shot)     : {before_n}/{len(TEST_TASKS)} correct ({before_n/len(TEST_TASKS):.0%})")
print(f"After calibration (0-shot) : {after_correct}/{len(TEST_TASKS)} correct ({after_correct/len(TEST_TASKS):.0%})")
print(f"Tasks recalibrated (❌→✅)  : +{flipped}")
if regressed:
    print(f"Tasks regressed    (✅→❌)  : -{regressed}")


TEST SET — AFTER TRAINING
ID     Type                            True   Baseline    After  Change
24     dyck_language                      A   B ❌      A ✅  ← RECALIBRATED ✨
34     boolean_expressions                A   B ❌      A ✅  ← RECALIBRATED ✨
65     boolean_expressions                A   B ❌      A ✅  ← RECALIBRATED ✨
74     navigate                           C   B ❌      C ✅  ← RECALIBRATED ✨
88     navigate                           B   C ❌      C ❌  
5      dyck_language                      A   B ❌      A ✅  ← RECALIBRATED ✨
30     navigate                           C   B ❌      C ✅  ← RECALIBRATED ✨
173    boolean_expressions                A   B ❌      A ✅  ← RECALIBRATED ✨

Baseline (pure 0-shot)     : 0/8 correct (0%)
After calibration (0-shot) : 7/8 correct (88%)
Tasks recalibrated (❌→✅)  : +7


In [13]:
print("Confidence Scores After Training:")
for t in TEST_TASKS:
    pred, prob = predict_mc(student_model, t["student_prompt"])
    mark = "✅" if pred == t["answer"] else "❌"
    print(f"Task {t['id']:3d} | Pred: {pred} | Confidence: {prob:.2f} {mark}")

Confidence Scores After Training:
Task  24 | Pred: A | Confidence: 0.00 ✅
Task  34 | Pred: A | Confidence: 0.00 ✅
Task  65 | Pred: A | Confidence: 0.00 ✅
Task  74 | Pred: C | Confidence: 0.00 ✅
Task  88 | Pred: C | Confidence: 0.00 ❌
Task   5 | Pred: A | Confidence: 0.00 ✅
Task  30 | Pred: C | Confidence: 0.00 ✅
Task 173 | Pred: A | Confidence: 0.00 ✅


In [17]:
all_tasks = r"""[
  {
    "id": 1,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nLeo has the map.\nRyan has the key.\nAlice has the book.\nSwap Leo and Ryan.\nSwap Leo and Alice.\nWho has the map?\nOptions:\nA) Alice\nB) Charlie\nC) Leo\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nLeo has the map.\nRyan has the key.\nAlice has the book.\nSwap Leo and Ryan.\nSwap Leo and Alice.\nWho has the map?\nOptions:\nA) Alice\nB) Charlie\nC) Leo\nD) Ryan",
    "answer": "D"
  },
  {
    "id": 2,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 3,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nRyan has the shell.\nLucy has the lamp.\nSwap Ryan and Lucy.\nWho has the lamp?\nOptions:\nA) Charlie\nB) David\nC) Lucy\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nRyan has the shell.\nLucy has the lamp.\nSwap Ryan and Lucy.\nWho has the lamp?\nOptions:\nA) Alice\nB) David\nC) Lucy\nD) Ryan",
    "answer": "D"
  },
  {
    "id": 4,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,2)\nWest 1\nSouth 5\nNorth 2\nOptions:\nA) (-1,2)\nB) (0,-1)\nC) (0,2)\nD) (3,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,2)\nWest 1\nSouth 5\nNorth 2\nOptions:\nA) (-1,2)\nB) (0,-1)\nC) (0,2)\nD) (3,2)",
    "answer": "B"
  },
  {
    "id": 5,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 6,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 7,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 8,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT False) OR False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT False) OR False\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 9,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()()()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()()()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 10,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(())()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(())()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 11,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 12,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 13,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,0)\nEast 2\nEast 1\nSouth 3\nWest 2\nEast 5\nOptions:\nA) (5,-4)\nB) (7,-5)\nC) (8,-3)\nD) (8,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,0)\nEast 2\nEast 1\nSouth 3\nWest 2\nEast 5\nOptions:\nA) (5,-4)\nB) (7,-5)\nC) (8,-3)\nD) (8,-6)",
    "answer": "C"
  },
  {
    "id": 14,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 15,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 16,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,1)\nNorth 4\nNorth 5\nNorth 2\nSouth 2\nEast 1\nOptions:\nA) (2,8)\nB) (3,10)\nC) (3,8)\nD) (4,13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,1)\nNorth 4\nNorth 5\nNorth 2\nSouth 2\nEast 1\nOptions:\nA) (2,8)\nB) (3,10)\nC) (3,8)\nD) (4,13)",
    "answer": "B"
  },
  {
    "id": 17,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nBen has the sword.\nCarl has the ball.\nSwap Ben and Carl.\nWho has the sword?\nOptions:\nA) Charlie\nB) Ben\nC) Carl\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nBen has the sword.\nCarl has the ball.\nSwap Ben and Carl.\nWho has the sword?\nOptions:\nA) Alice\nB) Ben\nC) Carl\nD) Peter",
    "answer": "C"
  },
  {
    "id": 18,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,5)\nEast 5\nNorth 2\nSouth 1\nEast 2\nOptions:\nA) (11,7)\nB) (5,5)\nC) (8,6)\nD) (9,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,5)\nEast 5\nNorth 2\nSouth 1\nEast 2\nOptions:\nA) (11,7)\nB) (5,5)\nC) (8,6)\nD) (9,6)",
    "answer": "C"
  },
  {
    "id": 19,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 20,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 21,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "answer": "D"
  },
  {
    "id": 22,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 23,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()(())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()(())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 24,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 25,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 4\nEast 4\nNorth 3\nNorth 4\nSouth 3\nOptions:\nA) (10,9)\nB) (7,13)\nC) (9,12)\nD) (9,13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 4\nEast 4\nNorth 3\nNorth 4\nSouth 3\nOptions:\nA) (10,9)\nB) (7,13)\nC) (9,12)\nD) (9,13)",
    "answer": "C"
  },
  {
    "id": 26,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,6)\nEast 2\nWest 4\nEast 4\nOptions:\nA) (4,3)\nB) (6,6)\nC) (7,3)\nD) (9,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,6)\nEast 2\nWest 4\nEast 4\nOptions:\nA) (4,3)\nB) (6,6)\nC) (7,3)\nD) (9,5)",
    "answer": "B"
  },
  {
    "id": 27,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()((())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()((())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 28,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 29,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,3)\nNorth 3\nSouth 5\nEast 3\nOptions:\nA) (3,1)\nB) (6,1)\nC) (8,0)\nD) (9,4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,3)\nNorth 3\nSouth 5\nEast 3\nOptions:\nA) (3,1)\nB) (6,1)\nC) (8,0)\nD) (9,4)",
    "answer": "B"
  },
  {
    "id": 30,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "answer": "C"
  },
  {
    "id": 31,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n))()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n))()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 32,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 33,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nKai has the badge.\nDavid has the pen.\nAnna has the notebook.\nSwap Kai and Anna.\nSwap Kai and David.\nWho has the pen?\nOptions:\nA) Anna\nB) David\nC) Kai\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nKai has the badge.\nDavid has the pen.\nAnna has the notebook.\nSwap Kai and Anna.\nSwap Kai and David.\nWho has the pen?\nOptions:\nA) Anna\nB) David\nC) Kai\nD) Sarah",
    "answer": "C"
  },
  {
    "id": 34,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 35,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nNorth 3\nSouth 1\nNorth 2\nEast 3\nOptions:\nA) (4,8)\nB) (5,9)\nC) (6,10)\nD) (6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nNorth 3\nSouth 1\nNorth 2\nEast 3\nOptions:\nA) (4,8)\nB) (5,9)\nC) (6,10)\nD) (6,6)",
    "answer": "B"
  },
  {
    "id": 36,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "answer": "C"
  },
  {
    "id": 37,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nCarl has the mirror.\nSarah has the notebook.\nSwap Carl and Sarah.\nWho has the notebook?\nOptions:\nA) Charlie\nB) Alice\nC) Carl\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nCarl has the mirror.\nSarah has the notebook.\nSwap Carl and Sarah.\nWho has the notebook?\nOptions:\nA) Bob\nB) Alice\nC) Carl\nD) Sarah",
    "answer": "C"
  },
  {
    "id": 38,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nBob has the pen.\nAmy has the wallet.\nLucy has the flask.\nSwap Amy and Lucy.\nSwap Bob and Amy.\nWho has the wallet?\nOptions:\nA) Amy\nB) Bob\nC) Lucy\nD) Tom",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nBob has the pen.\nAmy has the wallet.\nLucy has the flask.\nSwap Amy and Lucy.\nSwap Bob and Amy.\nWho has the wallet?\nOptions:\nA) Amy\nB) Bob\nC) Lucy\nD) Tom",
    "answer": "C"
  },
  {
    "id": 39,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nDavid has the ring.\nMike has the key.\nSwap David and Mike.\nWho has the key?\nOptions:\nA) Charlie\nB) Alice\nC) David\nD) Mike",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nDavid has the ring.\nMike has the key.\nSwap David and Mike.\nWho has the key?\nOptions:\nA) Bob\nB) Alice\nC) David\nD) Mike",
    "answer": "C"
  },
  {
    "id": 40,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,0)\nSouth 3\nWest 4\nEast 2\nSouth 2\nWest 5\nOptions:\nA) (-6,-2)\nB) (-6,-5)\nC) (-6,-6)\nD) (-8,-5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,0)\nSouth 3\nWest 4\nEast 2\nSouth 2\nWest 5\nOptions:\nA) (-6,-2)\nB) (-6,-5)\nC) (-6,-6)\nD) (-8,-5)",
    "answer": "B"
  },
  {
    "id": 41,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,3)\nWest 1\nEast 1\nWest 2\nWest 1\nWest 3\nOptions:\nA) (-3,0)\nB) (-4,3)\nC) (-6,3)\nD) (-6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,3)\nWest 1\nEast 1\nWest 2\nWest 1\nWest 3\nOptions:\nA) (-3,0)\nB) (-4,3)\nC) (-6,3)\nD) (-6,6)",
    "answer": "B"
  },
  {
    "id": 42,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,5)\nNorth 1\nSouth 4\nNorth 2\nOptions:\nA) (1,2)\nB) (1,6)\nC) (3,4)\nD) (3,7)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,5)\nNorth 1\nSouth 4\nNorth 2\nOptions:\nA) (1,2)\nB) (1,6)\nC) (3,4)\nD) (3,7)",
    "answer": "C"
  },
  {
    "id": 43,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,3)\nWest 5\nEast 4\nSouth 5\nOptions:\nA) (0,-4)\nB) (0,1)\nC) (3,-2)\nD) (4,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,3)\nWest 5\nEast 4\nSouth 5\nOptions:\nA) (0,-4)\nB) (0,1)\nC) (3,-2)\nD) (4,1)",
    "answer": "C"
  },
  {
    "id": 44,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nKate has the bag.\nOmar has the flask.\nFinn has the wallet.\nSwap Omar and Finn.\nSwap Kate and Finn.\nWho has the wallet?\nOptions:\nA) David\nB) Finn\nC) Kate\nD) Omar",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nKate has the bag.\nOmar has the flask.\nFinn has the wallet.\nSwap Omar and Finn.\nSwap Kate and Finn.\nWho has the wallet?\nOptions:\nA) David\nB) Finn\nC) Kate\nD) Omar",
    "answer": "D"
  },
  {
    "id": 45,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 46,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 47,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 48,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSarah has the key.\nIvy has the ball.\nPeter has the book.\nSwap Ivy and Peter.\nSwap Sarah and Ivy.\nWho has the key?\nOptions:\nA) Ivy\nB) John\nC) Peter\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSarah has the key.\nIvy has the ball.\nPeter has the book.\nSwap Ivy and Peter.\nSwap Sarah and Ivy.\nWho has the key?\nOptions:\nA) Ivy\nB) John\nC) Peter\nD) Sarah",
    "answer": "A"
  },
  {
    "id": 49,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 2\nWest 3\nEast 4\nWest 1\nOptions:\nA) (7,7)\nB) (8,3)\nC) (8,6)\nD) (9,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 2\nWest 3\nEast 4\nWest 1\nOptions:\nA) (7,7)\nB) (8,3)\nC) (8,6)\nD) (9,3)",
    "answer": "C"
  },
  {
    "id": 50,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAlice has the bag.\nJack has the cube.\nPeter has the glove.\nSwap Alice and Peter.\nSwap Alice and Jack.\nWho has the bag?\nOptions:\nA) Alice\nB) Finn\nC) Jack\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAlice has the bag.\nJack has the cube.\nPeter has the glove.\nSwap Alice and Peter.\nSwap Alice and Jack.\nWho has the bag?\nOptions:\nA) Alice\nB) Finn\nC) Jack\nD) Peter",
    "answer": "D"
  },
  {
    "id": 51,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,0)\nWest 3\nNorth 2\nEast 3\nOptions:\nA) (1,1)\nB) (1,3)\nC) (1,4)\nD) (2,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,0)\nWest 3\nNorth 2\nEast 3\nOptions:\nA) (1,1)\nB) (1,3)\nC) (1,4)\nD) (2,2)",
    "answer": "D"
  },
  {
    "id": 52,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "answer": "D"
  },
  {
    "id": 53,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,4)\nNorth 1\nNorth 2\nWest 2\nOptions:\nA) (0,7)\nB) (0,9)\nC) (1,4)\nD) (2,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,4)\nNorth 1\nNorth 2\nWest 2\nOptions:\nA) (0,7)\nB) (0,9)\nC) (1,4)\nD) (2,8)",
    "answer": "A"
  },
  {
    "id": 54,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 55,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 56,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 57,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "answer": "D"
  },
  {
    "id": 58,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nSouth 4\nWest 1\nNorth 3\nEast 3\nOptions:\nA) (3,2)\nB) (4,1)\nC) (5,-1)\nD) (8,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nSouth 4\nWest 1\nNorth 3\nEast 3\nOptions:\nA) (3,2)\nB) (4,1)\nC) (5,-1)\nD) (8,1)",
    "answer": "C"
  },
  {
    "id": 59,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,5)\nSouth 3\nWest 2\nWest 3\nWest 1\nNorth 5\nOptions:\nA) (-4,5)\nB) (-5,7)\nC) (-6,9)\nD) (-8,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,5)\nSouth 3\nWest 2\nWest 3\nWest 1\nNorth 5\nOptions:\nA) (-4,5)\nB) (-5,7)\nC) (-6,9)\nD) (-8,8)",
    "answer": "B"
  },
  {
    "id": 60,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 61,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,3)\nSouth 2\nSouth 2\nNorth 3\nOptions:\nA) (3,2)\nB) (4,4)\nC) (5,2)\nD) (5,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,3)\nSouth 2\nSouth 2\nNorth 3\nOptions:\nA) (3,2)\nB) (4,4)\nC) (5,2)\nD) (5,3)",
    "answer": "A"
  },
  {
    "id": 62,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 63,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 64,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nNorth 4\nNorth 4\nNorth 3\nOptions:\nA) (4,13)\nB) (5,13)\nC) (6,12)\nD) (7,14)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nNorth 4\nNorth 4\nNorth 3\nOptions:\nA) (4,13)\nB) (5,13)\nC) (6,12)\nD) (7,14)",
    "answer": "A"
  },
  {
    "id": 65,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 66,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nWest 1\nSouth 5\nWest 5\nOptions:\nA) (-1,-5)\nB) (2,-5)\nC) (2,-7)\nD) (2,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nWest 1\nSouth 5\nWest 5\nOptions:\nA) (-1,-5)\nB) (2,-5)\nC) (2,-7)\nD) (2,-8)",
    "answer": "A"
  },
  {
    "id": 67,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSam has the phone.\nLuke has the bag.\nAnna has the badge.\nSwap Sam and Anna.\nSwap Luke and Anna.\nWho has the badge?\nOptions:\nA) Anna\nB) Ivy\nC) Luke\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSam has the phone.\nLuke has the bag.\nAnna has the badge.\nSwap Sam and Anna.\nSwap Luke and Anna.\nWho has the badge?\nOptions:\nA) Anna\nB) Ivy\nC) Luke\nD) Sam",
    "answer": "D"
  },
  {
    "id": 68,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 69,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((((())))())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((((())))())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 70,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nMike has the ball.\nSarah has the key.\nAnna has the pen.\nSwap Mike and Anna.\nSwap Sarah and Anna.\nWho has the key?\nOptions:\nA) Anna\nB) Jack\nC) Mike\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nMike has the ball.\nSarah has the key.\nAnna has the pen.\nSwap Mike and Anna.\nSwap Sarah and Anna.\nWho has the key?\nOptions:\nA) Anna\nB) Jack\nC) Mike\nD) Sarah",
    "answer": "A"
  },
  {
    "id": 71,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nSouth 4\nSouth 3\nWest 2\nSouth 4\nOptions:\nA) (-2,-7)\nB) (-3,-10)\nC) (0,-9)\nD) (1,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nSouth 4\nSouth 3\nWest 2\nSouth 4\nOptions:\nA) (-2,-7)\nB) (-3,-10)\nC) (0,-9)\nD) (1,-6)",
    "answer": "A"
  },
  {
    "id": 72,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,3)\nWest 2\nSouth 5\nNorth 1\nSouth 1\nOptions:\nA) (-1,-2)\nB) (-1,-4)\nC) (-2,-3)\nD) (-4,-4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,3)\nWest 2\nSouth 5\nNorth 1\nSouth 1\nOptions:\nA) (-1,-2)\nB) (-1,-4)\nC) (-2,-3)\nD) (-4,-4)",
    "answer": "A"
  },
  {
    "id": 73,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND False) OR NOT (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND False) OR NOT (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 74,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "answer": "C"
  },
  {
    "id": 75,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nNorth 3\nWest 4\nSouth 3\nWest 2\nEast 4\nOptions:\nA) (-2,0)\nB) (-3,1)\nC) (-3,2)\nD) (-4,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nNorth 3\nWest 4\nSouth 3\nWest 2\nEast 4\nOptions:\nA) (-2,0)\nB) (-3,1)\nC) (-3,2)\nD) (-4,0)",
    "answer": "A"
  },
  {
    "id": 76,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 2\nWest 4\nNorth 4\nSouth 4\nOptions:\nA) (1,5)\nB) (2,7)\nC) (4,5)\nD) (4,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 2\nWest 4\nNorth 4\nSouth 4\nOptions:\nA) (1,5)\nB) (2,7)\nC) (4,5)\nD) (4,6)",
    "answer": "D"
  },
  {
    "id": 77,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nSouth 5\nNorth 4\nEast 4\nEast 3\nOptions:\nA) (11,3)\nB) (5,2)\nC) (6,7)\nD) (8,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nSouth 5\nNorth 4\nEast 4\nEast 3\nOptions:\nA) (11,3)\nB) (5,2)\nC) (6,7)\nD) (8,5)",
    "answer": "D"
  },
  {
    "id": 78,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nCharlie has the notebook.\nCarl has the laptop.\nJack has the watch.\nSwap Charlie and Jack.\nSwap Carl and Jack.\nWho has the notebook?\nOptions:\nA) Carl\nB) Charlie\nC) Jack\nD) Lucy",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nCharlie has the notebook.\nCarl has the laptop.\nJack has the watch.\nSwap Charlie and Jack.\nSwap Carl and Jack.\nWho has the notebook?\nOptions:\nA) Carl\nB) Charlie\nC) Jack\nD) Lucy",
    "answer": "A"
  },
  {
    "id": 79,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,3)\nSouth 4\nSouth 3\nNorth 2\nOptions:\nA) (3,-5)\nB) (3,0)\nC) (6,-2)\nD) (8,-2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,3)\nSouth 4\nSouth 3\nNorth 2\nOptions:\nA) (3,-5)\nB) (3,0)\nC) (6,-2)\nD) (8,-2)",
    "answer": "C"
  },
  {
    "id": 80,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n((False OR True) AND True) OR False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n((False OR True) AND True) OR False\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 81,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((((()()))))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((((()()))))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 82,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nNorth 3\nWest 2\nOptions:\nA) (10,1)\nB) (5,2)\nC) (7,1)\nD) (8,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nNorth 3\nWest 2\nOptions:\nA) (10,1)\nB) (5,2)\nC) (7,1)\nD) (8,1)",
    "answer": "D"
  },
  {
    "id": 83,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 84,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 85,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 86,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 87,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nTom has the phone.\nEmma has the glove.\nEva has the watch.\nSwap Tom and Emma.\nSwap Tom and Eva.\nWho has the watch?\nOptions:\nA) Emma\nB) Eva\nC) Tom\nD) Zara",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nTom has the phone.\nEmma has the glove.\nEva has the watch.\nSwap Tom and Emma.\nSwap Tom and Eva.\nWho has the watch?\nOptions:\nA) Emma\nB) Eva\nC) Tom\nD) Zara",
    "answer": "C"
  },
  {
    "id": 88,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "answer": "B"
  },
  {
    "id": 89,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nNorth 4\nWest 3\nNorth 5\nEast 3\nOptions:\nA) (2,11)\nB) (3,11)\nC) (4,9)\nD) (6,11)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nNorth 4\nWest 3\nNorth 5\nEast 3\nOptions:\nA) (2,11)\nB) (3,11)\nC) (4,9)\nD) (6,11)",
    "answer": "B"
  },
  {
    "id": 90,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()())(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()())(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 91,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,5)\nWest 5\nSouth 3\nEast 2\nEast 4\nOptions:\nA) (4,5)\nB) (6,2)\nC) (7,5)\nD) (8,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,5)\nWest 5\nSouth 3\nEast 2\nEast 4\nOptions:\nA) (4,5)\nB) (6,2)\nC) (7,5)\nD) (8,2)",
    "answer": "B"
  },
  {
    "id": 92,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 93,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nNorth 4\nNorth 3\nEast 4\nSouth 5\nSouth 1\nOptions:\nA) (2,-1)\nB) (3,0)\nC) (3,1)\nD) (5,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nNorth 4\nNorth 3\nEast 4\nSouth 5\nSouth 1\nOptions:\nA) (2,-1)\nB) (3,0)\nC) (3,1)\nD) (5,2)",
    "answer": "D"
  },
  {
    "id": 94,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,0)\nEast 2\nNorth 2\nEast 3\nOptions:\nA) (11,2)\nB) (12,4)\nC) (13,0)\nD) (9,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,0)\nEast 2\nNorth 2\nEast 3\nOptions:\nA) (11,2)\nB) (12,4)\nC) (13,0)\nD) (9,5)",
    "answer": "A"
  },
  {
    "id": 95,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "answer": "C"
  },
  {
    "id": 96,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 97,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 98,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 99,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nMike has the flask.\nPeter has the hat.\nCarl has the watch.\nSwap Mike and Carl.\nSwap Peter and Carl.\nWho has the hat?\nOptions:\nA) Carl\nB) Mike\nC) Peter\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nMike has the flask.\nPeter has the hat.\nCarl has the watch.\nSwap Mike and Carl.\nSwap Peter and Carl.\nWho has the hat?\nOptions:\nA) Carl\nB) Mike\nC) Peter\nD) Sarah",
    "answer": "A"
  },
  {
    "id": 100,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 101,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND True) OR NOT (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND True) OR NOT (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 102,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,2)\nSouth 5\nSouth 1\nEast 4\nWest 5\nSouth 2\nOptions:\nA) (2,-8)\nB) (3,-4)\nC) (5,-6)\nD) (6,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,2)\nSouth 5\nSouth 1\nEast 4\nWest 5\nSouth 2\nOptions:\nA) (2,-8)\nB) (3,-4)\nC) (5,-6)\nD) (6,-6)",
    "answer": "C"
  },
  {
    "id": 103,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nSouth 4\nWest 4\nSouth 4\nWest 5\nEast 5\nOptions:\nA) (-3,-6)\nB) (0,-3)\nC) (0,-5)\nD) (0,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nSouth 4\nWest 4\nSouth 4\nWest 5\nEast 5\nOptions:\nA) (-3,-6)\nB) (0,-3)\nC) (0,-5)\nD) (0,-6)",
    "answer": "D"
  },
  {
    "id": 104,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nEast 4\nSouth 2\nNorth 2\nOptions:\nA) (10,-1)\nB) (8,2)\nC) (8,5)\nD) (9,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nEast 4\nSouth 2\nNorth 2\nOptions:\nA) (10,-1)\nB) (8,2)\nC) (8,5)\nD) (9,3)",
    "answer": "B"
  },
  {
    "id": 105,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nEast 1\nOptions:\nA) (11,-2)\nB) (12,-3)\nC) (8,-1)\nD) (9,-4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nEast 1\nOptions:\nA) (11,-2)\nB) (12,-3)\nC) (8,-1)\nD) (9,-4)",
    "answer": "A"
  },
  {
    "id": 106,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 107,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,2)\nNorth 5\nWest 1\nEast 2\nSouth 1\nOptions:\nA) (2,3)\nB) (2,9)\nC) (3,6)\nD) (4,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,2)\nNorth 5\nWest 1\nEast 2\nSouth 1\nOptions:\nA) (2,3)\nB) (2,9)\nC) (3,6)\nD) (4,5)",
    "answer": "C"
  },
  {
    "id": 108,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nTom has the lamp.\nMia has the crown.\nSwap Tom and Mia.\nWho has the lamp?\nOptions:\nA) Charlie\nB) Carl\nC) Mia\nD) Tom",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nTom has the lamp.\nMia has the crown.\nSwap Tom and Mia.\nWho has the lamp?\nOptions:\nA) Alice\nB) Carl\nC) Mia\nD) Tom",
    "answer": "C"
  },
  {
    "id": 109,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,4)\nWest 3\nWest 3\nSouth 3\nSouth 5\nOptions:\nA) (-3,-3)\nB) (-5,-3)\nC) (-5,-4)\nD) (-8,-3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,4)\nWest 3\nWest 3\nSouth 3\nSouth 5\nOptions:\nA) (-3,-3)\nB) (-5,-3)\nC) (-5,-4)\nD) (-8,-3)",
    "answer": "C"
  },
  {
    "id": 110,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAmy has the notebook.\nSarah has the crown.\nAlice has the doll.\nSwap Amy and Sarah.\nSwap Amy and Alice.\nWho has the notebook?\nOptions:\nA) Alice\nB) Amy\nC) Kate\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAmy has the notebook.\nSarah has the crown.\nAlice has the doll.\nSwap Amy and Sarah.\nSwap Amy and Alice.\nWho has the notebook?\nOptions:\nA) Alice\nB) Amy\nC) Kate\nD) Sarah",
    "answer": "D"
  },
  {
    "id": 111,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,6)\nEast 3\nEast 4\nEast 4\nOptions:\nA) (13,3)\nB) (13,5)\nC) (14,9)\nD) (15,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,6)\nEast 3\nEast 4\nEast 4\nOptions:\nA) (13,3)\nB) (13,5)\nC) (14,9)\nD) (15,6)",
    "answer": "D"
  },
  {
    "id": 112,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,6)\nNorth 3\nWest 5\nNorth 1\nOptions:\nA) (-2,10)\nB) (-2,7)\nC) (-5,7)\nD) (-5,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,6)\nNorth 3\nWest 5\nNorth 1\nOptions:\nA) (-2,10)\nB) (-2,7)\nC) (-5,7)\nD) (-5,8)",
    "answer": "A"
  },
  {
    "id": 113,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nWest 2\nEast 2\nEast 1\nEast 2\nOptions:\nA) (5,5)\nB) (7,3)\nC) (7,7)\nD) (8,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nWest 2\nEast 2\nEast 1\nEast 2\nOptions:\nA) (5,5)\nB) (7,3)\nC) (7,7)\nD) (8,8)",
    "answer": "A"
  },
  {
    "id": 114,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSarah has the key.\nJohn has the ball.\nLuke has the doll.\nSwap Sarah and Luke.\nSwap Sarah and John.\nWho has the key?\nOptions:\nA) David\nB) John\nC) Luke\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSarah has the key.\nJohn has the ball.\nLuke has the doll.\nSwap Sarah and Luke.\nSwap Sarah and John.\nWho has the key?\nOptions:\nA) David\nB) John\nC) Luke\nD) Sarah",
    "answer": "C"
  },
  {
    "id": 115,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 116,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "answer": "C"
  },
  {
    "id": 117,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 118,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 119,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nCarl has the coin.\nKate has the pen.\nSwap Carl and Kate.\nWho has the pen?\nOptions:\nA) Charlie\nB) Carl\nC) Kate\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nCarl has the coin.\nKate has the pen.\nSwap Carl and Kate.\nWho has the pen?\nOptions:\nA) Alice\nB) Carl\nC) Kate\nD) Ryan",
    "answer": "B"
  },
  {
    "id": 120,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 5\nSouth 3\nNorth 1\nOptions:\nA) (11,4)\nB) (13,1)\nC) (8,7)\nD) (9,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 5\nSouth 3\nNorth 1\nOptions:\nA) (11,4)\nB) (13,1)\nC) (8,7)\nD) (9,6)",
    "answer": "A"
  },
  {
    "id": 121,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "answer": "B"
  },
  {
    "id": 122,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 123,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "answer": "C"
  },
  {
    "id": 124,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 125,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 126,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the sword.\nPeter has the mask.\nJack has the ball.\nSwap Emma and Jack.\nSwap Peter and Jack.\nWho has the mask?\nOptions:\nA) David\nB) Emma\nC) Jack\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the sword.\nPeter has the mask.\nJack has the ball.\nSwap Emma and Jack.\nSwap Peter and Jack.\nWho has the mask?\nOptions:\nA) David\nB) Emma\nC) Jack\nD) Peter",
    "answer": "C"
  },
  {
    "id": 127,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT ((True AND True) AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT ((True AND True) AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 128,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nWest 3\nNorth 2\nEast 2\nNorth 3\nNorth 1\nOptions:\nA) (0,9)\nB) (1,8)\nC) (1,9)\nD) (2,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nWest 3\nNorth 2\nEast 2\nNorth 3\nNorth 1\nOptions:\nA) (0,9)\nB) (1,8)\nC) (1,9)\nD) (2,8)",
    "answer": "D"
  },
  {
    "id": 129,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nPeter has the cube.\nEva has the hat.\nLucy has the mask.\nSwap Peter and Lucy.\nSwap Peter and Eva.\nWho has the hat?\nOptions:\nA) Eva\nB) Jack\nC) Lucy\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nPeter has the cube.\nEva has the hat.\nLucy has the mask.\nSwap Peter and Lucy.\nSwap Peter and Eva.\nWho has the hat?\nOptions:\nA) Eva\nB) Jack\nC) Lucy\nD) Peter",
    "answer": "D"
  },
  {
    "id": 130,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 131,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nOmar has the hat.\nIvy has the notebook.\nSwap Omar and Ivy.\nWho has the notebook?\nOptions:\nA) Charlie\nB) Ivy\nC) Luke\nD) Omar",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nOmar has the hat.\nIvy has the notebook.\nSwap Omar and Ivy.\nWho has the notebook?\nOptions:\nA) Alice\nB) Ivy\nC) Luke\nD) Omar",
    "answer": "D"
  },
  {
    "id": 132,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nSouth 5\nSouth 3\nNorth 1\nWest 1\nOptions:\nA) (1,-6)\nB) (4,-5)\nC) (6,-7)\nD) (7,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nSouth 5\nSouth 3\nNorth 1\nWest 1\nOptions:\nA) (1,-6)\nB) (4,-5)\nC) (6,-7)\nD) (7,-8)",
    "answer": "B"
  },
  {
    "id": 133,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nSouth 1\nEast 5\nEast 5\nSouth 4\nEast 2\nOptions:\nA) (13,-3)\nB) (16,-3)\nC) (16,0)\nD) (17,-4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nSouth 1\nEast 5\nEast 5\nSouth 4\nEast 2\nOptions:\nA) (13,-3)\nB) (16,-3)\nC) (16,0)\nD) (17,-4)",
    "answer": "B"
  },
  {
    "id": 134,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT ((False OR False) OR (True AND False))\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT ((False OR False) OR (True AND False))\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 135,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 136,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nJohn has the phone.\nSam has the crown.\nPeter has the wallet.\nSwap John and Sam.\nSwap John and Peter.\nWho has the phone?\nOptions:\nA) Amy\nB) John\nC) Peter\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nJohn has the phone.\nSam has the crown.\nPeter has the wallet.\nSwap John and Sam.\nSwap John and Peter.\nWho has the phone?\nOptions:\nA) Amy\nB) John\nC) Peter\nD) Sam",
    "answer": "D"
  },
  {
    "id": 137,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 138,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 139,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nSouth 2\nEast 4\nSouth 3\nNorth 3\nOptions:\nA) (2,0)\nB) (3,1)\nC) (4,-4)\nD) (5,-1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nSouth 2\nEast 4\nSouth 3\nNorth 3\nOptions:\nA) (2,0)\nB) (3,1)\nC) (4,-4)\nD) (5,-1)",
    "answer": "D"
  },
  {
    "id": 140,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 141,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Charlie\nB) Eva\nC) Kate\nD) Mia",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Alice\nB) Eva\nC) Kate\nD) Mia",
    "answer": "C"
  },
  {
    "id": 142,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nNorth 2\nEast 5\nNorth 4\nEast 2\nOptions:\nA) (10,6)\nB) (11,6)\nC) (7,8)\nD) (8,9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nNorth 2\nEast 5\nNorth 4\nEast 2\nOptions:\nA) (10,6)\nB) (11,6)\nC) (7,8)\nD) (8,9)",
    "answer": "A"
  },
  {
    "id": 143,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 144,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(50/5)-(3*2)\nOptions:\nA) 0\nB) 2\nC) 3\nD) 4",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(50/5)-(3*2)\nOptions:\nA) 0\nB) 2\nC) 3\nD) 4",
    "answer": "D"
  },
  {
    "id": 145,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 146,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "answer": "C"
  },
  {
    "id": 147,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,5)\nWest 4\nNorth 2\nSouth 3\nOptions:\nA) (-4,2)\nB) (-4,4)\nC) (-4,5)\nD) (-7,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,5)\nWest 4\nNorth 2\nSouth 3\nOptions:\nA) (-4,2)\nB) (-4,4)\nC) (-4,5)\nD) (-7,3)",
    "answer": "B"
  },
  {
    "id": 148,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nEast 2\nNorth 4\nWest 2\nEast 4\nOptions:\nA) (10,9)\nB) (5,6)\nC) (6,4)\nD) (7,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nEast 2\nNorth 4\nWest 2\nEast 4\nOptions:\nA) (10,9)\nB) (5,6)\nC) (6,4)\nD) (7,6)",
    "answer": "D"
  },
  {
    "id": 149,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the laptop.\nOmar has the glove.\nLucy has the map.\nSwap Emma and Omar.\nSwap Omar and Lucy.\nWho has the glove?\nOptions:\nA) Emma\nB) Lucy\nC) Omar\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the laptop.\nOmar has the glove.\nLucy has the map.\nSwap Emma and Omar.\nSwap Omar and Lucy.\nWho has the glove?\nOptions:\nA) Emma\nB) Lucy\nC) Omar\nD) Sam",
    "answer": "A"
  },
  {
    "id": 150,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nSouth 1\nSouth 1\nNorth 1\nOptions:\nA) (-3,-4)\nB) (-3,1)\nC) (0,-1)\nD) (3,-2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nSouth 1\nSouth 1\nNorth 1\nOptions:\nA) (-3,-4)\nB) (-3,1)\nC) (0,-1)\nD) (3,-2)",
    "answer": "C"
  },
  {
    "id": 151,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 2\nSouth 5\nNorth 5\nWest 5\nOptions:\nA) (-4,5)\nB) (-5,6)\nC) (-5,7)\nD) (-6,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 2\nSouth 5\nNorth 5\nWest 5\nOptions:\nA) (-4,5)\nB) (-5,6)\nC) (-5,7)\nD) (-6,8)",
    "answer": "B"
  },
  {
    "id": 152,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nEast 3\nWest 2\nSouth 2\nEast 1\nEast 2\nOptions:\nA) (2,1)\nB) (2,2)\nC) (4,2)\nD) (5,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nEast 3\nWest 2\nSouth 2\nEast 1\nEast 2\nOptions:\nA) (2,1)\nB) (2,2)\nC) (4,2)\nD) (5,3)",
    "answer": "C"
  },
  {
    "id": 153,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 154,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 155,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 156,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nEast 1\nSouth 3\nWest 3\nOptions:\nA) (-1,-2)\nB) (-2,1)\nC) (-2,3)\nD) (-3,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nEast 1\nSouth 3\nWest 3\nOptions:\nA) (-1,-2)\nB) (-2,1)\nC) (-2,3)\nD) (-3,1)",
    "answer": "B"
  },
  {
    "id": 157,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT True AND NOT False) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT True AND NOT False) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 158,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nKai has the guitar.\nKate has the camera.\nSwap Kai and Kate.\nWho has the guitar?\nOptions:\nA) Charlie\nB) David\nC) Kai\nD) Kate",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nKai has the guitar.\nKate has the camera.\nSwap Kai and Kate.\nWho has the guitar?\nOptions:\nA) Alice\nB) David\nC) Kai\nD) Kate",
    "answer": "D"
  },
  {
    "id": 159,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())))()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())))()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 160,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,6)\nSouth 2\nNorth 2\nEast 5\nWest 2\nOptions:\nA) (3,9)\nB) (5,5)\nC) (6,6)\nD) (9,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,6)\nSouth 2\nNorth 2\nEast 5\nWest 2\nOptions:\nA) (3,9)\nB) (5,5)\nC) (6,6)\nD) (9,8)",
    "answer": "C"
  },
  {
    "id": 161,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,3)\nSouth 3\nWest 5\nWest 2\nWest 3\nSouth 3\nOptions:\nA) (-11,-1)\nB) (-8,-3)\nC) (-9,-1)\nD) (-9,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,3)\nSouth 3\nWest 5\nWest 2\nWest 3\nSouth 3\nOptions:\nA) (-11,-1)\nB) (-8,-3)\nC) (-9,-1)\nD) (-9,0)",
    "answer": "B"
  },
  {
    "id": 162,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nSouth 2\nSouth 4\nSouth 5\nSouth 1\nOptions:\nA) (0,-15)\nB) (1,-12)\nC) (3,-12)\nD) (5,-9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nSouth 2\nSouth 4\nSouth 5\nSouth 1\nOptions:\nA) (0,-15)\nB) (1,-12)\nC) (3,-12)\nD) (5,-9)",
    "answer": "C"
  },
  {
    "id": 163,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,5)\nSouth 1\nEast 4\nNorth 4\nWest 3\nOptions:\nA) (-1,11)\nB) (1,8)\nC) (3,6)\nD) (4,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,5)\nSouth 1\nEast 4\nNorth 4\nWest 3\nOptions:\nA) (-1,11)\nB) (1,8)\nC) (3,6)\nD) (4,5)",
    "answer": "B"
  },
  {
    "id": 164,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "answer": "C"
  },
  {
    "id": 165,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 166,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,1)\nEast 5\nSouth 5\nSouth 5\nOptions:\nA) (5,-8)\nB) (6,-8)\nC) (7,-9)\nD) (8,-12)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,1)\nEast 5\nSouth 5\nSouth 5\nOptions:\nA) (5,-8)\nB) (6,-8)\nC) (7,-9)\nD) (8,-12)",
    "answer": "C"
  },
  {
    "id": 167,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nSouth 5\nSouth 3\nNorth 3\nWest 2\nEast 2\nOptions:\nA) (4,1)\nB) (6,1)\nC) (8,-2)\nD) (8,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nSouth 5\nSouth 3\nNorth 3\nWest 2\nEast 2\nOptions:\nA) (4,1)\nB) (6,1)\nC) (8,-2)\nD) (8,2)",
    "answer": "B"
  },
  {
    "id": 168,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n18/(3+3)\nOptions:\nA) -1\nB) 1\nC) 2\nD) 3",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n18/(3+3)\nOptions:\nA) -1\nB) 1\nC) 2\nD) 3",
    "answer": "D"
  },
  {
    "id": 169,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "answer": "C"
  },
  {
    "id": 170,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 171,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "answer": "C"
  },
  {
    "id": 172,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 173,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 174,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 175,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nSouth 2\nNorth 1\nWest 3\nSouth 2\nOptions:\nA) (-1,-4)\nB) (-3,-3)\nC) (-5,-1)\nD) (0,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nSouth 2\nNorth 1\nWest 3\nSouth 2\nOptions:\nA) (-1,-4)\nB) (-3,-3)\nC) (-5,-1)\nD) (0,0)",
    "answer": "B"
  },
  {
    "id": 176,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 177,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nLucy has the doll.\nCarl has the flask.\nDavid has the pen.\nSwap Lucy and David.\nSwap Lucy and Carl.\nWho has the flask?\nOptions:\nA) Carl\nB) David\nC) Lucy\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nLucy has the doll.\nCarl has the flask.\nDavid has the pen.\nSwap Lucy and David.\nSwap Lucy and Carl.\nWho has the flask?\nOptions:\nA) Carl\nB) David\nC) Lucy\nD) Sam",
    "answer": "C"
  },
  {
    "id": 178,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "answer": "B"
  },
  {
    "id": 179,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nSouth 3\nNorth 1\nSouth 2\nSouth 5\nNorth 1\nOptions:\nA) (2,-8)\nB) (4,-7)\nC) (6,-7)\nD) (7,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nSouth 3\nNorth 1\nSouth 2\nSouth 5\nNorth 1\nOptions:\nA) (2,-8)\nB) (4,-7)\nC) (6,-7)\nD) (7,-6)",
    "answer": "B"
  },
  {
    "id": 180,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "answer": "C"
  },
  {
    "id": 181,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,4)\nSouth 4\nNorth 4\nSouth 2\nNorth 4\nOptions:\nA) (-1,5)\nB) (0,3)\nC) (0,8)\nD) (1,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,4)\nSouth 4\nNorth 4\nSouth 2\nNorth 4\nOptions:\nA) (-1,5)\nB) (0,3)\nC) (0,8)\nD) (1,6)",
    "answer": "D"
  },
  {
    "id": 182,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n())(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n())(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 183,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nEast 2\nEast 2\nNorth 2\nEast 4\nOptions:\nA) (10,7)\nB) (11,4)\nC) (13,8)\nD) (7,10)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nEast 2\nEast 2\nNorth 2\nEast 4\nOptions:\nA) (10,7)\nB) (11,4)\nC) (13,8)\nD) (7,10)",
    "answer": "A"
  },
  {
    "id": 184,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,6)\nEast 4\nSouth 1\nNorth 1\nWest 1\nEast 2\nOptions:\nA) (4,8)\nB) (6,4)\nC) (6,8)\nD) (7,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,6)\nEast 4\nSouth 1\nNorth 1\nWest 1\nEast 2\nOptions:\nA) (4,8)\nB) (6,4)\nC) (6,8)\nD) (7,6)",
    "answer": "D"
  },
  {
    "id": 185,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False) AND (NOT True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False) AND (NOT True)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 186,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 187,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 188,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,0)\nSouth 5\nSouth 2\nNorth 2\nOptions:\nA) (0,-6)\nB) (1,-5)\nC) (1,-6)\nD) (4,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,0)\nSouth 5\nSouth 2\nNorth 2\nOptions:\nA) (0,-6)\nB) (1,-5)\nC) (1,-6)\nD) (4,-6)",
    "answer": "B"
  },
  {
    "id": 189,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR False) AND (True OR True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR False) AND (True OR True)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 190,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the map.\nBob has the shell.\nCarl has the ring.\nSwap Bob and Carl.\nSwap David and Bob.\nWho has the shell?\nOptions:\nA) Bob\nB) Carl\nC) David\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the map.\nBob has the shell.\nCarl has the ring.\nSwap Bob and Carl.\nSwap David and Bob.\nWho has the shell?\nOptions:\nA) Bob\nB) Carl\nC) David\nD) Ryan",
    "answer": "B"
  },
  {
    "id": 191,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the sword.\nLeo has the bag.\nSwap Eva and Leo.\nWho has the sword?\nOptions:\nA) Charlie\nB) Eva\nC) Finn\nD) Leo",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the sword.\nLeo has the bag.\nSwap Eva and Leo.\nWho has the sword?\nOptions:\nA) Alice\nB) Eva\nC) Finn\nD) Leo",
    "answer": "D"
  },
  {
    "id": 192,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nEast 4\nEast 5\nNorth 2\nWest 5\nWest 3\nOptions:\nA) (2,3)\nB) (2,4)\nC) (3,5)\nD) (4,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nEast 4\nEast 5\nNorth 2\nWest 5\nWest 3\nOptions:\nA) (2,3)\nB) (2,4)\nC) (3,5)\nD) (4,3)",
    "answer": "D"
  },
  {
    "id": 193,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 194,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 195,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 196,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "answer": "D"
  },
  {
    "id": 197,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,3)\nWest 2\nEast 4\nNorth 2\nSouth 5\nOptions:\nA) (0,-3)\nB) (1,-1)\nC) (3,-3)\nD) (3,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,3)\nWest 2\nEast 4\nNorth 2\nSouth 5\nOptions:\nA) (0,-3)\nB) (1,-1)\nC) (3,-3)\nD) (3,0)",
    "answer": "D"
  },
  {
    "id": 198,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR True) AND (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR True) AND (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 199,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 200,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,4)\nSouth 3\nSouth 1\nEast 5\nSouth 2\nOptions:\nA) (10,-2)\nB) (11,-1)\nC) (12,1)\nD) (9,-2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,4)\nSouth 3\nSouth 1\nEast 5\nSouth 2\nOptions:\nA) (10,-2)\nB) (11,-1)\nC) (12,1)\nD) (9,-2)",
    "answer": "D"
  }
]"""

In [22]:
import os
import torch
import json

print("="*80)
print("1. LOADING 200 TASKS & CHECKPOINT")
print("="*80)

# 200 tasks HARDCODED here so it doesn't try to open any files!
raw_tasks_json = r"""[
 {
  "id": 1,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nLeo has the map.\nRyan has the key.\nAlice has the book.\nSwap Leo and Ryan.\nSwap Leo and Alice.\nWho has the map?\nOptions:\nA) Alice\nB) Charlie\nC) Leo\nD) Ryan",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nLeo has the map.\nRyan has the key.\nAlice has the book.\nSwap Leo and Ryan.\nSwap Leo and Alice.\nWho has the map?\nOptions:\nA) Alice\nB) Charlie\nC) Leo\nD) Ryan",
  "answer": "D"
 },
 {
  "id": 2,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 3,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nRyan has the shell.\nLucy has the lamp.\nSwap Ryan and Lucy.\nWho has the lamp?\nOptions:\nA) Charlie\nB) David\nC) Lucy\nD) Ryan",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nRyan has the shell.\nLucy has the lamp.\nSwap Ryan and Lucy.\nWho has the lamp?\nOptions:\nA) Alice\nB) David\nC) Lucy\nD) Ryan",
  "answer": "D"
 },
 {
  "id": 4,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,2)\nWest 1\nSouth 5\nNorth 2\nOptions:\nA) (-1,2)\nB) (0,-1)\nC) (0,2)\nD) (3,2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,2)\nWest 1\nSouth 5\nNorth 2\nOptions:\nA) (-1,2)\nB) (0,-1)\nC) (0,2)\nD) (3,2)",
  "answer": "B"
 },
 {
  "id": 5,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 6,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 7,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 8,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT False) OR False\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT False) OR False\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 9,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()()()(\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()()()(\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 10,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(())()))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(())()))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 11,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 12,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 13,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,0)\nEast 2\nEast 1\nSouth 3\nWest 2\nEast 5\nOptions:\nA) (5,-4)\nB) (7,-5)\nC) (8,-3)\nD) (8,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,0)\nEast 2\nEast 1\nSouth 3\nWest 2\nEast 5\nOptions:\nA) (5,-4)\nB) (7,-5)\nC) (8,-3)\nD) (8,-6)",
  "answer": "C"
 },
 {
  "id": 14,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 15,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 16,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,1)\nNorth 4\nNorth 5\nNorth 2\nSouth 2\nEast 1\nOptions:\nA) (2,8)\nB) (3,10)\nC) (3,8)\nD) (4,13)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,1)\nNorth 4\nNorth 5\nNorth 2\nSouth 2\nEast 1\nOptions:\nA) (2,8)\nB) (3,10)\nC) (3,8)\nD) (4,13)",
  "answer": "B"
 },
 {
  "id": 17,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nBen has the sword.\nCarl has the ball.\nSwap Ben and Carl.\nWho has the sword?\nOptions:\nA) Charlie\nB) Ben\nC) Carl\nD) Peter",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nBen has the sword.\nCarl has the ball.\nSwap Ben and Carl.\nWho has the sword?\nOptions:\nA) Alice\nB) Ben\nC) Carl\nD) Peter",
  "answer": "C"
 },
 {
  "id": 18,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,5)\nEast 5\nNorth 2\nSouth 1\nEast 2\nOptions:\nA) (11,7)\nB) (5,5)\nC) (8,6)\nD) (9,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,5)\nEast 5\nNorth 2\nSouth 1\nEast 2\nOptions:\nA) (11,7)\nB) (5,5)\nC) (8,6)\nD) (9,6)",
  "answer": "C"
 },
 {
  "id": 19,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 20,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 21,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
  "answer": "D"
 },
 {
  "id": 22,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 23,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()(())()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()(())()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 24,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 25,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 4\nEast 4\nNorth 3\nNorth 4\nSouth 3\nOptions:\nA) (10,9)\nB) (7,13)\nC) (9,12)\nD) (9,13)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 4\nEast 4\nNorth 3\nNorth 4\nSouth 3\nOptions:\nA) (10,9)\nB) (7,13)\nC) (9,12)\nD) (9,13)",
  "answer": "C"
 },
 {
  "id": 26,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,6)\nEast 2\nWest 4\nEast 4\nOptions:\nA) (4,3)\nB) (6,6)\nC) (7,3)\nD) (9,5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,6)\nEast 2\nWest 4\nEast 4\nOptions:\nA) (4,3)\nB) (6,6)\nC) (7,3)\nD) (9,5)",
  "answer": "B"
 },
 {
  "id": 27,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()((())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()((())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 28,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()()()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()()()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 29,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,3)\nNorth 3\nSouth 5\nEast 3\nOptions:\nA) (3,1)\nB) (6,1)\nC) (8,0)\nD) (9,4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,3)\nNorth 3\nSouth 5\nEast 3\nOptions:\nA) (3,1)\nB) (6,1)\nC) (8,0)\nD) (9,4)",
  "answer": "B"
 },
 {
  "id": 30,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
  "answer": "C"
 },
 {
  "id": 31,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n))()(())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n))()(())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 32,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 33,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nKai has the badge.\nDavid has the pen.\nAnna has the notebook.\nSwap Kai and Anna.\nSwap Kai and David.\nWho has the pen?\nOptions:\nA) Anna\nB) David\nC) Kai\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nKai has the badge.\nDavid has the pen.\nAnna has the notebook.\nSwap Kai and Anna.\nSwap Kai and David.\nWho has the pen?\nOptions:\nA) Anna\nB) David\nC) Kai\nD) Sarah",
  "answer": "C"
 },
 {
  "id": 34,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 35,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nNorth 3\nSouth 1\nNorth 2\nEast 3\nOptions:\nA) (4,8)\nB) (5,9)\nC) (6,10)\nD) (6,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nNorth 3\nSouth 1\nNorth 2\nEast 3\nOptions:\nA) (4,8)\nB) (5,9)\nC) (6,10)\nD) (6,6)",
  "answer": "B"
 },
 {
  "id": 36,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
  "answer": "C"
 },
 {
  "id": 37,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nCarl has the mirror.\nSarah has the notebook.\nSwap Carl and Sarah.\nWho has the notebook?\nOptions:\nA) Charlie\nB) Alice\nC) Carl\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nCarl has the mirror.\nSarah has the notebook.\nSwap Carl and Sarah.\nWho has the notebook?\nOptions:\nA) Bob\nB) Alice\nC) Carl\nD) Sarah",
  "answer": "C"
 },
 {
  "id": 38,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nBob has the pen.\nAmy has the wallet.\nLucy has the flask.\nSwap Amy and Lucy.\nSwap Bob and Amy.\nWho has the wallet?\nOptions:\nA) Amy\nB) Bob\nC) Lucy\nD) Tom",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nBob has the pen.\nAmy has the wallet.\nLucy has the flask.\nSwap Amy and Lucy.\nSwap Bob and Amy.\nWho has the wallet?\nOptions:\nA) Amy\nB) Bob\nC) Lucy\nD) Tom",
  "answer": "C"
 },
 {
  "id": 39,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nDavid has the ring.\nMike has the key.\nSwap David and Mike.\nWho has the key?\nOptions:\nA) Charlie\nB) Alice\nC) David\nD) Mike",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nDavid has the ring.\nMike has the key.\nSwap David and Mike.\nWho has the key?\nOptions:\nA) Bob\nB) Alice\nC) David\nD) Mike",
  "answer": "C"
 },
 {
  "id": 40,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,0)\nSouth 3\nWest 4\nEast 2\nSouth 2\nWest 5\nOptions:\nA) (-6,-2)\nB) (-6,-5)\nC) (-6,-6)\nD) (-8,-5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,0)\nSouth 3\nWest 4\nEast 2\nSouth 2\nWest 5\nOptions:\nA) (-6,-2)\nB) (-6,-5)\nC) (-6,-6)\nD) (-8,-5)",
  "answer": "B"
 },
 {
  "id": 41,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,3)\nWest 1\nEast 1\nWest 2\nWest 1\nWest 3\nOptions:\nA) (-3,0)\nB) (-4,3)\nC) (-6,3)\nD) (-6,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,3)\nWest 1\nEast 1\nWest 2\nWest 1\nWest 3\nOptions:\nA) (-3,0)\nB) (-4,3)\nC) (-6,3)\nD) (-6,6)",
  "answer": "B"
 },
 {
  "id": 42,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,5)\nNorth 1\nSouth 4\nNorth 2\nOptions:\nA) (1,2)\nB) (1,6)\nC) (3,4)\nD) (3,7)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,5)\nNorth 1\nSouth 4\nNorth 2\nOptions:\nA) (1,2)\nB) (1,6)\nC) (3,4)\nD) (3,7)",
  "answer": "C"
 },
 {
  "id": 43,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,3)\nWest 5\nEast 4\nSouth 5\nOptions:\nA) (0,-4)\nB) (0,1)\nC) (3,-2)\nD) (4,1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,3)\nWest 5\nEast 4\nSouth 5\nOptions:\nA) (0,-4)\nB) (0,1)\nC) (3,-2)\nD) (4,1)",
  "answer": "C"
 },
 {
  "id": 44,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nKate has the bag.\nOmar has the flask.\nFinn has the wallet.\nSwap Omar and Finn.\nSwap Kate and Finn.\nWho has the wallet?\nOptions:\nA) David\nB) Finn\nC) Kate\nD) Omar",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nKate has the bag.\nOmar has the flask.\nFinn has the wallet.\nSwap Omar and Finn.\nSwap Kate and Finn.\nWho has the wallet?\nOptions:\nA) David\nB) Finn\nC) Kate\nD) Omar",
  "answer": "D"
 },
 {
  "id": 45,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 46,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 47,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 48,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSarah has the key.\nIvy has the ball.\nPeter has the book.\nSwap Ivy and Peter.\nSwap Sarah and Ivy.\nWho has the key?\nOptions:\nA) Ivy\nB) John\nC) Peter\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSarah has the key.\nIvy has the ball.\nPeter has the book.\nSwap Ivy and Peter.\nSwap Sarah and Ivy.\nWho has the key?\nOptions:\nA) Ivy\nB) John\nC) Peter\nD) Sarah",
  "answer": "A"
 },
 {
  "id": 49,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 2\nWest 3\nEast 4\nWest 1\nOptions:\nA) (7,7)\nB) (8,3)\nC) (8,6)\nD) (9,3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 2\nWest 3\nEast 4\nWest 1\nOptions:\nA) (7,7)\nB) (8,3)\nC) (8,6)\nD) (9,3)",
  "answer": "C"
 },
 {
  "id": 50,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAlice has the bag.\nJack has the cube.\nPeter has the glove.\nSwap Alice and Peter.\nSwap Alice and Jack.\nWho has the bag?\nOptions:\nA) Alice\nB) Finn\nC) Jack\nD) Peter",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAlice has the bag.\nJack has the cube.\nPeter has the glove.\nSwap Alice and Peter.\nSwap Alice and Jack.\nWho has the bag?\nOptions:\nA) Alice\nB) Finn\nC) Jack\nD) Peter",
  "answer": "D"
 },
 {
  "id": 51,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,0)\nWest 3\nNorth 2\nEast 3\nOptions:\nA) (1,1)\nB) (1,3)\nC) (1,4)\nD) (2,2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,0)\nWest 3\nNorth 2\nEast 3\nOptions:\nA) (1,1)\nB) (1,3)\nC) (1,4)\nD) (2,2)",
  "answer": "D"
 },
 {
  "id": 52,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
  "answer": "D"
 },
 {
  "id": 53,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,4)\nNorth 1\nNorth 2\nWest 2\nOptions:\nA) (0,7)\nB) (0,9)\nC) (1,4)\nD) (2,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,4)\nNorth 1\nNorth 2\nWest 2\nOptions:\nA) (0,7)\nB) (0,9)\nC) (1,4)\nD) (2,8)",
  "answer": "A"
 },
 {
  "id": 54,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 55,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 56,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 57,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
  "answer": "D"
 },
 {
  "id": 58,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nSouth 4\nWest 1\nNorth 3\nEast 3\nOptions:\nA) (3,2)\nB) (4,1)\nC) (5,-1)\nD) (8,1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nSouth 4\nWest 1\nNorth 3\nEast 3\nOptions:\nA) (3,2)\nB) (4,1)\nC) (5,-1)\nD) (8,1)",
  "answer": "C"
 },
 {
  "id": 59,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,5)\nSouth 3\nWest 2\nWest 3\nWest 1\nNorth 5\nOptions:\nA) (-4,5)\nB) (-5,7)\nC) (-6,9)\nD) (-8,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,5)\nSouth 3\nWest 2\nWest 3\nWest 1\nNorth 5\nOptions:\nA) (-4,5)\nB) (-5,7)\nC) (-6,9)\nD) (-8,8)",
  "answer": "B"
 },
 {
  "id": 60,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 61,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,3)\nSouth 2\nSouth 2\nNorth 3\nOptions:\nA) (3,2)\nB) (4,4)\nC) (5,2)\nD) (5,3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,3)\nSouth 2\nSouth 2\nNorth 3\nOptions:\nA) (3,2)\nB) (4,4)\nC) (5,2)\nD) (5,3)",
  "answer": "A"
 },
 {
  "id": 62,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 63,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 64,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nNorth 4\nNorth 4\nNorth 3\nOptions:\nA) (4,13)\nB) (5,13)\nC) (6,12)\nD) (7,14)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nNorth 4\nNorth 4\nNorth 3\nOptions:\nA) (4,13)\nB) (5,13)\nC) (6,12)\nD) (7,14)",
  "answer": "A"
 },
 {
  "id": 65,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 66,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nWest 1\nSouth 5\nWest 5\nOptions:\nA) (-1,-5)\nB) (2,-5)\nC) (2,-7)\nD) (2,-8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nWest 1\nSouth 5\nWest 5\nOptions:\nA) (-1,-5)\nB) (2,-5)\nC) (2,-7)\nD) (2,-8)",
  "answer": "A"
 },
 {
  "id": 67,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSam has the phone.\nLuke has the bag.\nAnna has the badge.\nSwap Sam and Anna.\nSwap Luke and Anna.\nWho has the badge?\nOptions:\nA) Anna\nB) Ivy\nC) Luke\nD) Sam",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSam has the phone.\nLuke has the bag.\nAnna has the badge.\nSwap Sam and Anna.\nSwap Luke and Anna.\nWho has the badge?\nOptions:\nA) Anna\nB) Ivy\nC) Luke\nD) Sam",
  "answer": "D"
 },
 {
  "id": 68,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 69,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((((())))())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((((())))())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 70,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nMike has the ball.\nSarah has the key.\nAnna has the pen.\nSwap Mike and Anna.\nSwap Sarah and Anna.\nWho has the key?\nOptions:\nA) Anna\nB) Jack\nC) Mike\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nMike has the ball.\nSarah has the key.\nAnna has the pen.\nSwap Mike and Anna.\nSwap Sarah and Anna.\nWho has the key?\nOptions:\nA) Anna\nB) Jack\nC) Mike\nD) Sarah",
  "answer": "A"
 },
 {
  "id": 71,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nSouth 4\nSouth 3\nWest 2\nSouth 4\nOptions:\nA) (-2,-7)\nB) (-3,-10)\nC) (0,-9)\nD) (1,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nSouth 4\nSouth 3\nWest 2\nSouth 4\nOptions:\nA) (-2,-7)\nB) (-3,-10)\nC) (0,-9)\nD) (1,-6)",
  "answer": "A"
 },
 {
  "id": 72,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,3)\nWest 2\nSouth 5\nNorth 1\nSouth 1\nOptions:\nA) (-1,-2)\nB) (-1,-4)\nC) (-2,-3)\nD) (-4,-4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,3)\nWest 2\nSouth 5\nNorth 1\nSouth 1\nOptions:\nA) (-1,-2)\nB) (-1,-4)\nC) (-2,-3)\nD) (-4,-4)",
  "answer": "A"
 },
 {
  "id": 73,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND False) OR NOT (True AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND False) OR NOT (True AND False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 74,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
  "answer": "C"
 },
 {
  "id": 75,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nNorth 3\nWest 4\nSouth 3\nWest 2\nEast 4\nOptions:\nA) (-2,0)\nB) (-3,1)\nC) (-3,2)\nD) (-4,0)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nNorth 3\nWest 4\nSouth 3\nWest 2\nEast 4\nOptions:\nA) (-2,0)\nB) (-3,1)\nC) (-3,2)\nD) (-4,0)",
  "answer": "A"
 },
 {
  "id": 76,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 2\nWest 4\nNorth 4\nSouth 4\nOptions:\nA) (1,5)\nB) (2,7)\nC) (4,5)\nD) (4,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 2\nWest 4\nNorth 4\nSouth 4\nOptions:\nA) (1,5)\nB) (2,7)\nC) (4,5)\nD) (4,6)",
  "answer": "D"
 },
 {
  "id": 77,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nSouth 5\nNorth 4\nEast 4\nEast 3\nOptions:\nA) (11,3)\nB) (5,2)\nC) (6,7)\nD) (8,5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nSouth 5\nNorth 4\nEast 4\nEast 3\nOptions:\nA) (11,3)\nB) (5,2)\nC) (6,7)\nD) (8,5)",
  "answer": "D"
 },
 {
  "id": 78,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nCharlie has the notebook.\nCarl has the laptop.\nJack has the watch.\nSwap Charlie and Jack.\nSwap Carl and Jack.\nWho has the notebook?\nOptions:\nA) Carl\nB) Charlie\nC) Jack\nD) Lucy",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nCharlie has the notebook.\nCarl has the laptop.\nJack has the watch.\nSwap Charlie and Jack.\nSwap Carl and Jack.\nWho has the notebook?\nOptions:\nA) Carl\nB) Charlie\nC) Jack\nD) Lucy",
  "answer": "A"
 },
 {
  "id": 79,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,3)\nSouth 4\nSouth 3\nNorth 2\nOptions:\nA) (3,-5)\nB) (3,0)\nC) (6,-2)\nD) (8,-2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,3)\nSouth 4\nSouth 3\nNorth 2\nOptions:\nA) (3,-5)\nB) (3,0)\nC) (6,-2)\nD) (8,-2)",
  "answer": "C"
 },
 {
  "id": 80,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n((False OR True) AND True) OR False\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n((False OR True) AND True) OR False\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 81,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((((()()))))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((((()()))))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 82,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nNorth 3\nWest 2\nOptions:\nA) (10,1)\nB) (5,2)\nC) (7,1)\nD) (8,1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nNorth 3\nWest 2\nOptions:\nA) (10,1)\nB) (5,2)\nC) (7,1)\nD) (8,1)",
  "answer": "D"
 },
 {
  "id": 83,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 84,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 85,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 86,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
  "answer": "D"
 },
 {
  "id": 87,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nTom has the phone.\nEmma has the glove.\nEva has the watch.\nSwap Tom and Emma.\nSwap Tom and Eva.\nWho has the watch?\nOptions:\nA) Emma\nB) Eva\nC) Tom\nD) Zara",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nTom has the phone.\nEmma has the glove.\nEva has the watch.\nSwap Tom and Emma.\nSwap Tom and Eva.\nWho has the watch?\nOptions:\nA) Emma\nB) Eva\nC) Tom\nD) Zara",
  "answer": "C"
 },
 {
  "id": 88,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
  "answer": "B"
 },
 {
  "id": 89,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nNorth 4\nWest 3\nNorth 5\nEast 3\nOptions:\nA) (2,11)\nB) (3,11)\nC) (4,9)\nD) (6,11)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nNorth 4\nWest 3\nNorth 5\nEast 3\nOptions:\nA) (2,11)\nB) (3,11)\nC) (4,9)\nD) (6,11)",
  "answer": "B"
 },
 {
  "id": 90,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()())(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()())(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 91,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,5)\nWest 5\nSouth 3\nEast 2\nEast 4\nOptions:\nA) (4,5)\nB) (6,2)\nC) (7,5)\nD) (8,2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,5)\nWest 5\nSouth 3\nEast 2\nEast 4\nOptions:\nA) (4,5)\nB) (6,2)\nC) (7,5)\nD) (8,2)",
  "answer": "B"
 },
 {
  "id": 92,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 93,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nNorth 4\nNorth 3\nEast 4\nSouth 5\nSouth 1\nOptions:\nA) (2,-1)\nB) (3,0)\nC) (3,1)\nD) (5,2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nNorth 4\nNorth 3\nEast 4\nSouth 5\nSouth 1\nOptions:\nA) (2,-1)\nB) (3,0)\nC) (3,1)\nD) (5,2)",
  "answer": "D"
 },
 {
  "id": 94,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,0)\nEast 2\nNorth 2\nEast 3\nOptions:\nA) (11,2)\nB) (12,4)\nC) (13,0)\nD) (9,5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,0)\nEast 2\nNorth 2\nEast 3\nOptions:\nA) (11,2)\nB) (12,4)\nC) (13,0)\nD) (9,5)",
  "answer": "A"
 },
 {
  "id": 95,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
  "answer": "C"
 },
 {
  "id": 96,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 97,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()(())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()(())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 98,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 99,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nMike has the flask.\nPeter has the hat.\nCarl has the watch.\nSwap Mike and Carl.\nSwap Peter and Carl.\nWho has the hat?\nOptions:\nA) Carl\nB) Mike\nC) Peter\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nMike has the flask.\nPeter has the hat.\nCarl has the watch.\nSwap Mike and Carl.\nSwap Peter and Carl.\nWho has the hat?\nOptions:\nA) Carl\nB) Mike\nC) Peter\nD) Sarah",
  "answer": "A"
 },
 {
  "id": 100,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())()(\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())()(\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 101,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND True) OR NOT (False AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND True) OR NOT (False AND True)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 102,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,2)\nSouth 5\nSouth 1\nEast 4\nWest 5\nSouth 2\nOptions:\nA) (2,-8)\nB) (3,-4)\nC) (5,-6)\nD) (6,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,2)\nSouth 5\nSouth 1\nEast 4\nWest 5\nSouth 2\nOptions:\nA) (2,-8)\nB) (3,-4)\nC) (5,-6)\nD) (6,-6)",
  "answer": "C"
 },
 {
  "id": 103,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nSouth 4\nWest 4\nSouth 4\nWest 5\nEast 5\nOptions:\nA) (-3,-6)\nB) (0,-3)\nC) (0,-5)\nD) (0,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nSouth 4\nWest 4\nSouth 4\nWest 5\nEast 5\nOptions:\nA) (-3,-6)\nB) (0,-3)\nC) (0,-5)\nD) (0,-6)",
  "answer": "D"
 },
 {
  "id": 104,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nEast 4\nSouth 2\nNorth 2\nOptions:\nA) (10,-1)\nB) (8,2)\nC) (8,5)\nD) (9,3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nEast 4\nSouth 2\nNorth 2\nOptions:\nA) (10,-1)\nB) (8,2)\nC) (8,5)\nD) (9,3)",
  "answer": "B"
 },
 {
  "id": 105,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nEast 1\nOptions:\nA) (11,-2)\nB) (12,-3)\nC) (8,-1)\nD) (9,-4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nEast 1\nOptions:\nA) (11,-2)\nB) (12,-3)\nC) (8,-1)\nD) (9,-4)",
  "answer": "A"
 },
 {
  "id": 106,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 107,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,2)\nNorth 5\nWest 1\nEast 2\nSouth 1\nOptions:\nA) (2,3)\nB) (2,9)\nC) (3,6)\nD) (4,5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,2)\nNorth 5\nWest 1\nEast 2\nSouth 1\nOptions:\nA) (2,3)\nB) (2,9)\nC) (3,6)\nD) (4,5)",
  "answer": "C"
 },
 {
  "id": 108,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nTom has the lamp.\nMia has the crown.\nSwap Tom and Mia.\nWho has the lamp?\nOptions:\nA) Charlie\nB) Carl\nC) Mia\nD) Tom",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nTom has the lamp.\nMia has the crown.\nSwap Tom and Mia.\nWho has the lamp?\nOptions:\nA) Alice\nB) Carl\nC) Mia\nD) Tom",
  "answer": "C"
 },
 {
  "id": 109,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,4)\nWest 3\nWest 3\nSouth 3\nSouth 5\nOptions:\nA) (-3,-3)\nB) (-5,-3)\nC) (-5,-4)\nD) (-8,-3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,4)\nWest 3\nWest 3\nSouth 3\nSouth 5\nOptions:\nA) (-3,-3)\nB) (-5,-3)\nC) (-5,-4)\nD) (-8,-3)",
  "answer": "C"
 },
 {
  "id": 110,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAmy has the notebook.\nSarah has the crown.\nAlice has the doll.\nSwap Amy and Sarah.\nSwap Amy and Alice.\nWho has the notebook?\nOptions:\nA) Alice\nB) Amy\nC) Kate\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAmy has the notebook.\nSarah has the crown.\nAlice has the doll.\nSwap Amy and Sarah.\nSwap Amy and Alice.\nWho has the notebook?\nOptions:\nA) Alice\nB) Amy\nC) Kate\nD) Sarah",
  "answer": "D"
 },
 {
  "id": 111,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,6)\nEast 3\nEast 4\nEast 4\nOptions:\nA) (13,3)\nB) (13,5)\nC) (14,9)\nD) (15,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,6)\nEast 3\nEast 4\nEast 4\nOptions:\nA) (13,3)\nB) (13,5)\nC) (14,9)\nD) (15,6)",
  "answer": "D"
 },
 {
  "id": 112,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,6)\nNorth 3\nWest 5\nNorth 1\nOptions:\nA) (-2,10)\nB) (-2,7)\nC) (-5,7)\nD) (-5,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,6)\nNorth 3\nWest 5\nNorth 1\nOptions:\nA) (-2,10)\nB) (-2,7)\nC) (-5,7)\nD) (-5,8)",
  "answer": "A"
 },
 {
  "id": 113,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nWest 2\nEast 2\nEast 1\nEast 2\nOptions:\nA) (5,5)\nB) (7,3)\nC) (7,7)\nD) (8,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nWest 2\nEast 2\nEast 1\nEast 2\nOptions:\nA) (5,5)\nB) (7,3)\nC) (7,7)\nD) (8,8)",
  "answer": "A"
 },
 {
  "id": 114,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSarah has the key.\nJohn has the ball.\nLuke has the doll.\nSwap Sarah and Luke.\nSwap Sarah and John.\nWho has the key?\nOptions:\nA) David\nB) John\nC) Luke\nD) Sarah",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSarah has the key.\nJohn has the ball.\nLuke has the doll.\nSwap Sarah and Luke.\nSwap Sarah and John.\nWho has the key?\nOptions:\nA) David\nB) John\nC) Luke\nD) Sarah",
  "answer": "C"
 },
 {
  "id": 115,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 116,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
  "answer": "C"
 },
 {
  "id": 117,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 118,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 119,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nCarl has the coin.\nKate has the pen.\nSwap Carl and Kate.\nWho has the pen?\nOptions:\nA) Charlie\nB) Carl\nC) Kate\nD) Ryan",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nCarl has the coin.\nKate has the pen.\nSwap Carl and Kate.\nWho has the pen?\nOptions:\nA) Alice\nB) Carl\nC) Kate\nD) Ryan",
  "answer": "B"
 },
 {
  "id": 120,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 5\nSouth 3\nNorth 1\nOptions:\nA) (11,4)\nB) (13,1)\nC) (8,7)\nD) (9,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 5\nSouth 3\nNorth 1\nOptions:\nA) (11,4)\nB) (13,1)\nC) (8,7)\nD) (9,6)",
  "answer": "A"
 },
 {
  "id": 121,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
  "answer": "B"
 },
 {
  "id": 122,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 123,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
  "answer": "C"
 },
 {
  "id": 124,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 125,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 126,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the sword.\nPeter has the mask.\nJack has the ball.\nSwap Emma and Jack.\nSwap Peter and Jack.\nWho has the mask?\nOptions:\nA) David\nB) Emma\nC) Jack\nD) Peter",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the sword.\nPeter has the mask.\nJack has the ball.\nSwap Emma and Jack.\nSwap Peter and Jack.\nWho has the mask?\nOptions:\nA) David\nB) Emma\nC) Jack\nD) Peter",
  "answer": "C"
 },
 {
  "id": 127,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT ((True AND True) AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT ((True AND True) AND False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 128,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nWest 3\nNorth 2\nEast 2\nNorth 3\nNorth 1\nOptions:\nA) (0,9)\nB) (1,8)\nC) (1,9)\nD) (2,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nWest 3\nNorth 2\nEast 2\nNorth 3\nNorth 1\nOptions:\nA) (0,9)\nB) (1,8)\nC) (1,9)\nD) (2,8)",
  "answer": "D"
 },
 {
  "id": 129,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nPeter has the cube.\nEva has the hat.\nLucy has the mask.\nSwap Peter and Lucy.\nSwap Peter and Eva.\nWho has the hat?\nOptions:\nA) Eva\nB) Jack\nC) Lucy\nD) Peter",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nPeter has the cube.\nEva has the hat.\nLucy has the mask.\nSwap Peter and Lucy.\nSwap Peter and Eva.\nWho has the hat?\nOptions:\nA) Eva\nB) Jack\nC) Lucy\nD) Peter",
  "answer": "D"
 },
 {
  "id": 130,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
  "answer": "D"
 },
 {
  "id": 131,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nOmar has the hat.\nIvy has the notebook.\nSwap Omar and Ivy.\nWho has the notebook?\nOptions:\nA) Charlie\nB) Ivy\nC) Luke\nD) Omar",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nOmar has the hat.\nIvy has the notebook.\nSwap Omar and Ivy.\nWho has the notebook?\nOptions:\nA) Alice\nB) Ivy\nC) Luke\nD) Omar",
  "answer": "D"
 },
 {
  "id": 132,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nSouth 5\nSouth 3\nNorth 1\nWest 1\nOptions:\nA) (1,-6)\nB) (4,-5)\nC) (6,-7)\nD) (7,-8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nSouth 5\nSouth 3\nNorth 1\nWest 1\nOptions:\nA) (1,-6)\nB) (4,-5)\nC) (6,-7)\nD) (7,-8)",
  "answer": "B"
 },
 {
  "id": 133,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nSouth 1\nEast 5\nEast 5\nSouth 4\nEast 2\nOptions:\nA) (13,-3)\nB) (16,-3)\nC) (16,0)\nD) (17,-4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nSouth 1\nEast 5\nEast 5\nSouth 4\nEast 2\nOptions:\nA) (13,-3)\nB) (16,-3)\nC) (16,0)\nD) (17,-4)",
  "answer": "B"
 },
 {
  "id": 134,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT ((False OR False) OR (True AND False))\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT ((False OR False) OR (True AND False))\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 135,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 136,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nJohn has the phone.\nSam has the crown.\nPeter has the wallet.\nSwap John and Sam.\nSwap John and Peter.\nWho has the phone?\nOptions:\nA) Amy\nB) John\nC) Peter\nD) Sam",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nJohn has the phone.\nSam has the crown.\nPeter has the wallet.\nSwap John and Sam.\nSwap John and Peter.\nWho has the phone?\nOptions:\nA) Amy\nB) John\nC) Peter\nD) Sam",
  "answer": "D"
 },
 {
  "id": 137,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 138,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 139,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nSouth 2\nEast 4\nSouth 3\nNorth 3\nOptions:\nA) (2,0)\nB) (3,1)\nC) (4,-4)\nD) (5,-1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nSouth 2\nEast 4\nSouth 3\nNorth 3\nOptions:\nA) (2,0)\nB) (3,1)\nC) (4,-4)\nD) (5,-1)",
  "answer": "D"
 },
 {
  "id": 140,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 141,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Charlie\nB) Eva\nC) Kate\nD) Mia",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Alice\nB) Eva\nC) Kate\nD) Mia",
  "answer": "C"
 },
 {
  "id": 142,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nNorth 2\nEast 5\nNorth 4\nEast 2\nOptions:\nA) (10,6)\nB) (11,6)\nC) (7,8)\nD) (8,9)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nNorth 2\nEast 5\nNorth 4\nEast 2\nOptions:\nA) (10,6)\nB) (11,6)\nC) (7,8)\nD) (8,9)",
  "answer": "A"
 },
 {
  "id": 143,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 144,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(50/5)-(3*2)\nOptions:\nA) 0\nB) 2\nC) 3\nD) 4",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(50/5)-(3*2)\nOptions:\nA) 0\nB) 2\nC) 3\nD) 4",
  "answer": "D"
 },
 {
  "id": 145,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 146,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
  "answer": "C"
 },
 {
  "id": 147,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,5)\nWest 4\nNorth 2\nSouth 3\nOptions:\nA) (-4,2)\nB) (-4,4)\nC) (-4,5)\nD) (-7,3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,5)\nWest 4\nNorth 2\nSouth 3\nOptions:\nA) (-4,2)\nB) (-4,4)\nC) (-4,5)\nD) (-7,3)",
  "answer": "B"
 },
 {
  "id": 148,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nEast 2\nNorth 4\nWest 2\nEast 4\nOptions:\nA) (10,9)\nB) (5,6)\nC) (6,4)\nD) (7,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nEast 2\nNorth 4\nWest 2\nEast 4\nOptions:\nA) (10,9)\nB) (5,6)\nC) (6,4)\nD) (7,6)",
  "answer": "D"
 },
 {
  "id": 149,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the laptop.\nOmar has the glove.\nLucy has the map.\nSwap Emma and Omar.\nSwap Omar and Lucy.\nWho has the glove?\nOptions:\nA) Emma\nB) Lucy\nC) Omar\nD) Sam",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the laptop.\nOmar has the glove.\nLucy has the map.\nSwap Emma and Omar.\nSwap Omar and Lucy.\nWho has the glove?\nOptions:\nA) Emma\nB) Lucy\nC) Omar\nD) Sam",
  "answer": "A"
 },
 {
  "id": 150,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nSouth 1\nSouth 1\nNorth 1\nOptions:\nA) (-3,-4)\nB) (-3,1)\nC) (0,-1)\nD) (3,-2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nSouth 1\nSouth 1\nNorth 1\nOptions:\nA) (-3,-4)\nB) (-3,1)\nC) (0,-1)\nD) (3,-2)",
  "answer": "C"
 },
 {
  "id": 151,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 2\nSouth 5\nNorth 5\nWest 5\nOptions:\nA) (-4,5)\nB) (-5,6)\nC) (-5,7)\nD) (-6,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 2\nSouth 5\nNorth 5\nWest 5\nOptions:\nA) (-4,5)\nB) (-5,6)\nC) (-5,7)\nD) (-6,8)",
  "answer": "B"
 },
 {
  "id": 152,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nEast 3\nWest 2\nSouth 2\nEast 1\nEast 2\nOptions:\nA) (2,1)\nB) (2,2)\nC) (4,2)\nD) (5,3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nEast 3\nWest 2\nSouth 2\nEast 1\nEast 2\nOptions:\nA) (2,1)\nB) (2,2)\nC) (4,2)\nD) (5,3)",
  "answer": "C"
 },
 {
  "id": 153,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 154,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 155,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 156,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nEast 1\nSouth 3\nWest 3\nOptions:\nA) (-1,-2)\nB) (-2,1)\nC) (-2,3)\nD) (-3,1)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nEast 1\nSouth 3\nWest 3\nOptions:\nA) (-1,-2)\nB) (-2,1)\nC) (-2,3)\nD) (-3,1)",
  "answer": "B"
 },
 {
  "id": 157,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT True AND NOT False) OR True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT True AND NOT False) OR True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 158,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nKai has the guitar.\nKate has the camera.\nSwap Kai and Kate.\nWho has the guitar?\nOptions:\nA) Charlie\nB) David\nC) Kai\nD) Kate",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nKai has the guitar.\nKate has the camera.\nSwap Kai and Kate.\nWho has the guitar?\nOptions:\nA) Alice\nB) David\nC) Kai\nD) Kate",
  "answer": "D"
 },
 {
  "id": 159,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())))()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())))()()\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 160,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,6)\nSouth 2\nNorth 2\nEast 5\nWest 2\nOptions:\nA) (3,9)\nB) (5,5)\nC) (6,6)\nD) (9,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,6)\nSouth 2\nNorth 2\nEast 5\nWest 2\nOptions:\nA) (3,9)\nB) (5,5)\nC) (6,6)\nD) (9,8)",
  "answer": "C"
 },
 {
  "id": 161,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,3)\nSouth 3\nWest 5\nWest 2\nWest 3\nSouth 3\nOptions:\nA) (-11,-1)\nB) (-8,-3)\nC) (-9,-1)\nD) (-9,0)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,3)\nSouth 3\nWest 5\nWest 2\nWest 3\nSouth 3\nOptions:\nA) (-11,-1)\nB) (-8,-3)\nC) (-9,-1)\nD) (-9,0)",
  "answer": "B"
 },
 {
  "id": 162,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nSouth 2\nSouth 4\nSouth 5\nSouth 1\nOptions:\nA) (0,-15)\nB) (1,-12)\nC) (3,-12)\nD) (5,-9)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nSouth 2\nSouth 4\nSouth 5\nSouth 1\nOptions:\nA) (0,-15)\nB) (1,-12)\nC) (3,-12)\nD) (5,-9)",
  "answer": "C"
 },
 {
  "id": 163,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,5)\nSouth 1\nEast 4\nNorth 4\nWest 3\nOptions:\nA) (-1,11)\nB) (1,8)\nC) (3,6)\nD) (4,5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,5)\nSouth 1\nEast 4\nNorth 4\nWest 3\nOptions:\nA) (-1,11)\nB) (1,8)\nC) (3,6)\nD) (4,5)",
  "answer": "B"
 },
 {
  "id": 164,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
  "answer": "C"
 },
 {
  "id": 165,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 166,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,1)\nEast 5\nSouth 5\nSouth 5\nOptions:\nA) (5,-8)\nB) (6,-8)\nC) (7,-9)\nD) (8,-12)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,1)\nEast 5\nSouth 5\nSouth 5\nOptions:\nA) (5,-8)\nB) (6,-8)\nC) (7,-9)\nD) (8,-12)",
  "answer": "C"
 },
 {
  "id": 167,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nSouth 5\nSouth 3\nNorth 3\nWest 2\nEast 2\nOptions:\nA) (4,1)\nB) (6,1)\nC) (8,-2)\nD) (8,2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nSouth 5\nSouth 3\nNorth 3\nWest 2\nEast 2\nOptions:\nA) (4,1)\nB) (6,1)\nC) (8,-2)\nD) (8,2)",
  "answer": "B"
 },
 {
  "id": 168,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n18/(3+3)\nOptions:\nA) -1\nB) 1\nC) 2\nD) 3",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n18/(3+3)\nOptions:\nA) -1\nB) 1\nC) 2\nD) 3",
  "answer": "D"
 },
 {
  "id": 169,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
  "answer": "C"
 },
 {
  "id": 170,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 171,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
  "answer": "C"
 },
 {
  "id": 172,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 173,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 174,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
  "answer": "D"
 },
 {
  "id": 175,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nSouth 2\nNorth 1\nWest 3\nSouth 2\nOptions:\nA) (-1,-4)\nB) (-3,-3)\nC) (-5,-1)\nD) (0,0)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nSouth 2\nNorth 1\nWest 3\nSouth 2\nOptions:\nA) (-1,-4)\nB) (-3,-3)\nC) (-5,-1)\nD) (0,0)",
  "answer": "B"
 },
 {
  "id": 176,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A"
 },
 {
  "id": 177,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nLucy has the doll.\nCarl has the flask.\nDavid has the pen.\nSwap Lucy and David.\nSwap Lucy and Carl.\nWho has the flask?\nOptions:\nA) Carl\nB) David\nC) Lucy\nD) Sam",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nLucy has the doll.\nCarl has the flask.\nDavid has the pen.\nSwap Lucy and David.\nSwap Lucy and Carl.\nWho has the flask?\nOptions:\nA) Carl\nB) David\nC) Lucy\nD) Sam",
  "answer": "C"
 },
 {
  "id": 178,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
  "answer": "B"
 },
 {
  "id": 179,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nSouth 3\nNorth 1\nSouth 2\nSouth 5\nNorth 1\nOptions:\nA) (2,-8)\nB) (4,-7)\nC) (6,-7)\nD) (7,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nSouth 3\nNorth 1\nSouth 2\nSouth 5\nNorth 1\nOptions:\nA) (2,-8)\nB) (4,-7)\nC) (6,-7)\nD) (7,-6)",
  "answer": "B"
 },
 {
  "id": 180,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
  "answer": "C"
 },
 {
  "id": 181,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,4)\nSouth 4\nNorth 4\nSouth 2\nNorth 4\nOptions:\nA) (-1,5)\nB) (0,3)\nC) (0,8)\nD) (1,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,4)\nSouth 4\nNorth 4\nSouth 2\nNorth 4\nOptions:\nA) (-1,5)\nB) (0,3)\nC) (0,8)\nD) (1,6)",
  "answer": "D"
 },
 {
  "id": 182,
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n())(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n())(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B"
 },
 {
  "id": 183,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nEast 2\nEast 2\nNorth 2\nEast 4\nOptions:\nA) (10,7)\nB) (11,4)\nC) (13,8)\nD) (7,10)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nEast 2\nEast 2\nNorth 2\nEast 4\nOptions:\nA) (10,7)\nB) (11,4)\nC) (13,8)\nD) (7,10)",
  "answer": "A"
 },
 {
  "id": 184,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,6)\nEast 4\nSouth 1\nNorth 1\nWest 1\nEast 2\nOptions:\nA) (4,8)\nB) (6,4)\nC) (6,8)\nD) (7,6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,6)\nEast 4\nSouth 1\nNorth 1\nWest 1\nEast 2\nOptions:\nA) (4,8)\nB) (6,4)\nC) (6,8)\nD) (7,6)",
  "answer": "D"
 },
 {
  "id": 185,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False) AND (NOT True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False) AND (NOT True)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 186,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 187,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 188,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,0)\nSouth 5\nSouth 2\nNorth 2\nOptions:\nA) (0,-6)\nB) (1,-5)\nC) (1,-6)\nD) (4,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,0)\nSouth 5\nSouth 2\nNorth 2\nOptions:\nA) (0,-6)\nB) (1,-5)\nC) (1,-6)\nD) (4,-6)",
  "answer": "B"
 },
 {
  "id": 189,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR False) AND (True OR True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR False) AND (True OR True)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 190,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the map.\nBob has the shell.\nCarl has the ring.\nSwap Bob and Carl.\nSwap David and Bob.\nWho has the shell?\nOptions:\nA) Bob\nB) Carl\nC) David\nD) Ryan",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the map.\nBob has the shell.\nCarl has the ring.\nSwap Bob and Carl.\nSwap David and Bob.\nWho has the shell?\nOptions:\nA) Bob\nB) Carl\nC) David\nD) Ryan",
  "answer": "B"
 },
 {
  "id": 191,
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the sword.\nLeo has the bag.\nSwap Eva and Leo.\nWho has the sword?\nOptions:\nA) Charlie\nB) Eva\nC) Finn\nD) Leo",
  "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the sword.\nLeo has the bag.\nSwap Eva and Leo.\nWho has the sword?\nOptions:\nA) Alice\nB) Eva\nC) Finn\nD) Leo",
  "answer": "D"
 },
 {
  "id": 192,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nEast 4\nEast 5\nNorth 2\nWest 5\nWest 3\nOptions:\nA) (2,3)\nB) (2,4)\nC) (3,5)\nD) (4,3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nEast 4\nEast 5\nNorth 2\nWest 5\nWest 3\nOptions:\nA) (2,3)\nB) (2,4)\nC) (3,5)\nD) (4,3)",
  "answer": "D"
 },
 {
  "id": 193,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 194,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
  "answer": "D"
 },
 {
  "id": 195,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR False)\nOptions:\nA) False\nB) True",
  "answer": "B"
 },
 {
  "id": 196,
  "task": "multistep_arithmetic",
  "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
  "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
  "answer": "D"
 },
 {
  "id": 197,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,3)\nWest 2\nEast 4\nNorth 2\nSouth 5\nOptions:\nA) (0,-3)\nB) (1,-1)\nC) (3,-3)\nD) (3,0)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,3)\nWest 2\nEast 4\nNorth 2\nSouth 5\nOptions:\nA) (0,-3)\nB) (1,-1)\nC) (3,-3)\nD) (3,0)",
  "answer": "D"
 },
 {
  "id": 198,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR True) AND (False AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR True) AND (False AND True)\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 199,
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
  "answer": "A"
 },
 {
  "id": 200,
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,4)\nSouth 3\nSouth 1\nEast 5\nSouth 2\nOptions:\nA) (10,-2)\nB) (11,-1)\nC) (12,1)\nD) (9,-2)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,4)\nSouth 3\nSouth 1\nEast 5\nSouth 2\nOptions:\nA) (10,-2)\nB) (11,-1)\nC) (12,1)\nD) (9,-2)",
  "answer": "D"
 }
]"""
all_tasks = json.loads(raw_tasks_json)

checkpoint_path = "phase22_adapter_checkpoint.pt"
ckpt = torch.load(checkpoint_path, map_location=device)

def set_adapter_weights(weights_list):
    for p, saved in zip(trainable_params, weights_list):
        p.data.copy_(saved.to(device))

def disable_adapter():
    zeros = [torch.zeros_like(p) for p in trainable_params]
    set_adapter_weights(zeros)

print("\n" + "="*80)
print("2. EVALUATING ALL 200 TASKS (THIS WILL TAKE ~2 MINUTES)")
print("="*80)

teacher_model.eval()
student_model.eval()

teacher_correct = 0
baseline_correct = 0
recal_correct = 0

print(f"{'ID':<4} | {'Type':<25} | {'True':<4} | {'Teacher (4s)':<12} | {'Base (0s)':<10} | {'Recal (0s)':<10}")
print("-" * 80)

for t in all_tasks:
    ans = t["answer"]
    
    t_pred, _ = predict_mc(teacher_model, t["teacher_prompt"])
    
    disable_adapter()
    b_pred, _ = predict_mc(student_model, t["student_prompt"])
    
    set_adapter_weights(ckpt["adapter_weights"])
    r_pred, _ = predict_mc(student_model, t["student_prompt"])
    
    if t_pred == ans: teacher_correct += 1
    if b_pred == ans: baseline_correct += 1
    if r_pred == ans: recal_correct += 1
        
    t_mark = "✅" if t_pred == ans else "❌"
    b_mark = "✅" if b_pred == ans else "❌"
    r_mark = "✅" if r_pred == ans else "❌"
    
    print(f"{t['id']:<4} | {t['task']:<25} | {ans:<4} | {t_pred} {t_mark:<10} | {b_pred} {b_mark:<8} | {r_pred} {r_mark}")

print("\n" + "="*80)
print("3. FINAL SCORES (OUT OF 200)")
print("="*80)
print(f"Teacher (4-shot)           : {teacher_correct}/200  ({teacher_correct/200:.1%})")
print(f"Student Baseline (0-shot)  : {baseline_correct}/200  ({baseline_correct/200:.1%})")
print(f"Student Recalibrated (0-s) : {recal_correct}/200  ({recal_correct/200:.1%})")


1. LOADING 200 TASKS & CHECKPOINT

2. EVALUATING ALL 200 TASKS (THIS WILL TAKE ~2 MINUTES)
ID   | Type                      | True | Teacher (4s) | Base (0s)  | Recal (0s)
--------------------------------------------------------------------------------
1    | tracking_shuffled_objects | D    | C ❌          | C ❌        | C ❌
2    | dyck_language             | A    | B ❌          | B ❌        | A ✅
3    | tracking_shuffled_objects | D    | C ❌          | C ❌        | C ❌
4    | navigate                  | B    | C ❌          | C ❌        | C ❌
5    | dyck_language             | A    | B ❌          | B ❌        | A ✅
6    | boolean_expressions       | A    | A ✅          | B ❌        | A ✅
7    | boolean_expressions       | B    | B ✅          | B ✅        | A ❌
8    | boolean_expressions       | B    | B ✅          | B ✅        | A ❌
9    | dyck_language             | B    | B ✅          | B ✅        | A ❌
10   | dyck_language             | B    | B ✅          | B ✅        | A ❌
11   | 

In [23]:
import os
import torch
import json

print("="*80)
print("1. LOADING 50 COMPLEX TASKS & CHECKPOINT")
print("="*80)

# 50 DEEP tasks HARDCODED here! Zero training contamination.
raw_tasks_json = r"""[
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A",
  "id": 1
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) AND (NOT True AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) AND (NOT True AND False)\nOptions:\nA) False\nB) True",
  "answer": "A",
  "id": 2
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR True) OR (NOT True OR True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR True) OR (NOT True OR True)\nOptions:\nA) False\nB) True",
  "answer": "B",
  "id": 3
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B",
  "id": 4
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the ring.\nCharlie has the key.\nGrace has the pen.\nBob has the book.\nSwap Bob and Charlie.\nSwap Bob and Grace.\nSwap Grace and Charlie.\nSwap Charlie and David.\nWho has the pen?\nOptions:\nA) David\nB) Charlie\nC) Grace\nD) Bob",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the ring.\nCharlie has the key.\nGrace has the pen.\nBob has the book.\nSwap Bob and Charlie.\nSwap Bob and Grace.\nSwap Grace and Charlie.\nSwap Charlie and David.\nWho has the pen?\nOptions:\nA) David\nB) Charlie\nC) Grace\nD) Bob",
  "answer": "D",
  "id": 5
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,3)\nSouth 3\nSouth 1\nWest 1\nWest 2\nSouth 4\nEast 3\nOptions:\nA) (-2,-8)\nB) (3,-2)\nC) (0,-5)\nD) (3,-7)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,3)\nSouth 3\nSouth 1\nWest 1\nWest 2\nSouth 4\nEast 3\nOptions:\nA) (-2,-8)\nB) (3,-2)\nC) (0,-5)\nD) (3,-7)",
  "answer": "C",
  "id": 6
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the ring.\nCharlie has the book.\nDavid has the key.\nGrace has the ball.\nSwap David and Charlie.\nSwap Emma and Grace.\nSwap Grace and Charlie.\nSwap Emma and David.\nWho has the ring?\nOptions:\nA) Charlie\nB) Emma\nC) David\nD) Grace",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the ring.\nCharlie has the book.\nDavid has the key.\nGrace has the ball.\nSwap David and Charlie.\nSwap Emma and Grace.\nSwap Grace and Charlie.\nSwap Emma and David.\nWho has the ring?\nOptions:\nA) Charlie\nB) Emma\nC) David\nD) Grace",
  "answer": "A",
  "id": 7
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B",
  "id": 8
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B",
  "id": 9
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,2)\nEast 5\nSouth 2\nWest 4\nSouth 5\nEast 4\nNorth 3\nOptions:\nA) (6,-1)\nB) (5,-2)\nC) (5,1)\nD) (4,-4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,2)\nEast 5\nSouth 2\nWest 4\nSouth 5\nEast 4\nNorth 3\nOptions:\nA) (6,-1)\nB) (5,-2)\nC) (5,1)\nD) (4,-4)",
  "answer": "B",
  "id": 10
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nNorth 2\nSouth 2\nWest 2\nSouth 1\nWest 4\nEast 5\nOptions:\nA) (2,-1)\nB) (-1,-1)\nC) (-1,2)\nD) (-4,-3)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nNorth 2\nSouth 2\nWest 2\nSouth 1\nWest 4\nEast 5\nOptions:\nA) (2,-1)\nB) (-1,-1)\nC) (-1,2)\nD) (-4,-3)",
  "answer": "B",
  "id": 11
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nBob has the ball.\nGrace has the key.\nEmma has the map.\nCharlie has the ring.\nSwap Grace and Emma.\nSwap Charlie and Grace.\nSwap Charlie and Grace.\nSwap Charlie and Grace.\nWho has the ball?\nOptions:\nA) Bob\nB) Grace\nC) Emma\nD) Charlie",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nBob has the ball.\nGrace has the key.\nEmma has the map.\nCharlie has the ring.\nSwap Grace and Emma.\nSwap Charlie and Grace.\nSwap Charlie and Grace.\nSwap Charlie and Grace.\nWho has the ball?\nOptions:\nA) Bob\nB) Grace\nC) Emma\nD) Charlie",
  "answer": "A",
  "id": 12
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,7)\nEast 4\nWest 5\nWest 2\nWest 4\nEast 2\nEast 5\nOptions:\nA) (4,6)\nB) (6,9)\nC) (6,7)\nD) (6,4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,7)\nEast 4\nWest 5\nWest 2\nWest 4\nEast 2\nEast 5\nOptions:\nA) (4,6)\nB) (6,9)\nC) (6,7)\nD) (6,4)",
  "answer": "C",
  "id": 13
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) OR (NOT False OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) OR (NOT False OR False)\nOptions:\nA) False\nB) True",
  "answer": "B",
  "id": 14
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND True) OR (NOT True OR False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND True) OR (NOT True OR False)\nOptions:\nA) False\nB) True",
  "answer": "A",
  "id": 15
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((()))()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((()))()())\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A",
  "id": 16
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,-10)\nEast 3\nWest 4\nSouth 4\nSouth 3\nWest 4\nNorth 4\nOptions:\nA) (-1,-11)\nB) (0,-13)\nC) (0,-11)\nD) (2,-10)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,-10)\nEast 3\nWest 4\nSouth 4\nSouth 3\nWest 4\nNorth 4\nOptions:\nA) (-1,-11)\nB) (0,-13)\nC) (0,-11)\nD) (2,-10)",
  "answer": "B",
  "id": 17
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) AND (NOT True OR True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) AND (NOT True OR True)\nOptions:\nA) False\nB) True",
  "answer": "B",
  "id": 18
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) AND (NOT False OR True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) AND (NOT False OR True)\nOptions:\nA) False\nB) True",
  "answer": "A",
  "id": 19
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B",
  "id": 20
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (10,8)\nSouth 4\nWest 4\nSouth 2\nNorth 1\nWest 2\nSouth 5\nOptions:\nA) (4,-2)\nB) (7,-5)\nC) (5,-4)\nD) (4,-5)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (10,8)\nSouth 4\nWest 4\nSouth 2\nNorth 1\nWest 2\nSouth 5\nOptions:\nA) (4,-2)\nB) (7,-5)\nC) (5,-4)\nD) (4,-5)",
  "answer": "A",
  "id": 21
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (-6,10)\nEast 4\nEast 1\nNorth 4\nNorth 1\nSouth 5\nEast 2\nOptions:\nA) (1,10)\nB) (-1,9)\nC) (0,8)\nD) (0,7)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (-6,10)\nEast 4\nEast 1\nNorth 4\nNorth 1\nSouth 5\nEast 2\nOptions:\nA) (1,10)\nB) (-1,9)\nC) (0,8)\nD) (0,7)",
  "answer": "A",
  "id": 22
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the book.\nBob has the ball.\nGrace has the ring.\nEmma has the map.\nSwap Bob and Emma.\nSwap Bob and Emma.\nSwap Grace and Bob.\nSwap Grace and Emma.\nWho has the ring?\nOptions:\nA) David\nB) Emma\nC) Grace\nD) Bob",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the book.\nBob has the ball.\nGrace has the ring.\nEmma has the map.\nSwap Bob and Emma.\nSwap Bob and Emma.\nSwap Grace and Bob.\nSwap Grace and Emma.\nWho has the ring?\nOptions:\nA) David\nB) Emma\nC) Grace\nD) Bob",
  "answer": "D",
  "id": 23
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) AND (NOT False AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) AND (NOT False AND True)\nOptions:\nA) False\nB) True",
  "answer": "B",
  "id": 24
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,-4)\nWest 2\nNorth 3\nEast 1\nSouth 3\nSouth 2\nSouth 1\nOptions:\nA) (3,-7)\nB) (0,-9)\nC) (6,-4)\nD) (3,-6)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,-4)\nWest 2\nNorth 3\nEast 1\nSouth 3\nSouth 2\nSouth 1\nOptions:\nA) (3,-7)\nB) (0,-9)\nC) (6,-4)\nD) (3,-6)",
  "answer": "A",
  "id": 25
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (9,10)\nNorth 5\nEast 1\nSouth 3\nNorth 1\nSouth 3\nEast 5\nOptions:\nA) (15,10)\nB) (13,12)\nC) (14,8)\nD) (17,12)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (9,10)\nNorth 5\nEast 1\nSouth 3\nNorth 1\nSouth 3\nEast 5\nOptions:\nA) (15,10)\nB) (13,12)\nC) (14,8)\nD) (17,12)",
  "answer": "A",
  "id": 26
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B",
  "id": 27
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A",
  "id": 28
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAlice has the ball.\nFrank has the coin.\nBob has the pen.\nCharlie has the key.\nSwap Frank and Alice.\nSwap Charlie and Frank.\nSwap Bob and Alice.\nSwap Frank and Charlie.\nWho has the coin?\nOptions:\nA) Alice\nB) Frank\nC) Bob\nD) Charlie",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAlice has the ball.\nFrank has the coin.\nBob has the pen.\nCharlie has the key.\nSwap Frank and Alice.\nSwap Charlie and Frank.\nSwap Bob and Alice.\nSwap Frank and Charlie.\nWho has the coin?\nOptions:\nA) Alice\nB) Frank\nC) Bob\nD) Charlie",
  "answer": "C",
  "id": 29
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nGrace has the map.\nFrank has the pen.\nEmma has the ball.\nAlice has the book.\nSwap Alice and Emma.\nSwap Emma and Grace.\nSwap Emma and Grace.\nSwap Alice and Frank.\nWho has the map?\nOptions:\nA) Grace\nB) Frank\nC) Emma\nD) Alice",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nGrace has the map.\nFrank has the pen.\nEmma has the ball.\nAlice has the book.\nSwap Alice and Emma.\nSwap Emma and Grace.\nSwap Emma and Grace.\nSwap Alice and Frank.\nWho has the map?\nOptions:\nA) Grace\nB) Frank\nC) Emma\nD) Alice",
  "answer": "A",
  "id": 30
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (-2,-9)\nNorth 4\nEast 1\nNorth 3\nSouth 3\nSouth 4\nWest 5\nOptions:\nA) (-9,-12)\nB) (-9,-7)\nC) (-8,-8)\nD) (-6,-9)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (-2,-9)\nNorth 4\nEast 1\nNorth 3\nSouth 3\nSouth 4\nWest 5\nOptions:\nA) (-9,-12)\nB) (-9,-7)\nC) (-8,-8)\nD) (-6,-9)",
  "answer": "D",
  "id": 31
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR False) OR (NOT True AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR False) OR (NOT True AND True)\nOptions:\nA) False\nB) True",
  "answer": "A",
  "id": 32
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A",
  "id": 33
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nGrace has the map.\nCharlie has the key.\nBob has the book.\nAlice has the ball.\nSwap Grace and Charlie.\nSwap Alice and Charlie.\nSwap Charlie and Grace.\nSwap Charlie and Grace.\nWho has the book?\nOptions:\nA) Grace\nB) Charlie\nC) Alice\nD) Bob",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nGrace has the map.\nCharlie has the key.\nBob has the book.\nAlice has the ball.\nSwap Grace and Charlie.\nSwap Alice and Charlie.\nSwap Charlie and Grace.\nSwap Charlie and Grace.\nWho has the book?\nOptions:\nA) Grace\nB) Charlie\nC) Alice\nD) Bob",
  "answer": "D",
  "id": 34
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A",
  "id": 35
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,-6)\nNorth 3\nEast 1\nEast 2\nSouth 1\nEast 5\nWest 5\nOptions:\nA) (9,-6)\nB) (4,-1)\nC) (8,-6)\nD) (6,-4)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,-6)\nNorth 3\nEast 1\nEast 2\nSouth 1\nEast 5\nWest 5\nOptions:\nA) (9,-6)\nB) (4,-1)\nC) (8,-6)\nD) (6,-4)",
  "answer": "D",
  "id": 36
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND False) OR (NOT True OR True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND False) OR (NOT True OR True)\nOptions:\nA) False\nB) True",
  "answer": "B",
  "id": 37
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nCharlie has the ring.\nDavid has the book.\nGrace has the coin.\nBob has the map.\nSwap Grace and David.\nSwap Grace and Charlie.\nSwap Bob and Charlie.\nSwap Charlie and David.\nWho has the book?\nOptions:\nA) Bob\nB) David\nC) Grace\nD) Charlie",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nCharlie has the ring.\nDavid has the book.\nGrace has the coin.\nBob has the map.\nSwap Grace and David.\nSwap Grace and Charlie.\nSwap Bob and Charlie.\nSwap Charlie and David.\nWho has the book?\nOptions:\nA) Bob\nB) David\nC) Grace\nD) Charlie",
  "answer": "A",
  "id": 38
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR False) AND (NOT True AND True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR False) AND (NOT True AND True)\nOptions:\nA) False\nB) True",
  "answer": "A",
  "id": 39
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the coin.\nFrank has the key.\nAlice has the book.\nBob has the ball.\nSwap Frank and Emma.\nSwap Frank and Alice.\nSwap Emma and Bob.\nSwap Emma and Frank.\nWho has the ball?\nOptions:\nA) Emma\nB) Frank\nC) Alice\nD) Bob",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the coin.\nFrank has the key.\nAlice has the book.\nBob has the ball.\nSwap Frank and Emma.\nSwap Frank and Alice.\nSwap Emma and Bob.\nSwap Emma and Frank.\nWho has the ball?\nOptions:\nA) Emma\nB) Frank\nC) Alice\nD) Bob",
  "answer": "B",
  "id": 40
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())(()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "B",
  "id": 41
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (-2,-8)\nEast 3\nWest 5\nEast 1\nNorth 3\nSouth 5\nEast 1\nOptions:\nA) (-5,-9)\nB) (-2,-11)\nC) (0,-7)\nD) (-2,-10)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (-2,-8)\nEast 3\nWest 5\nEast 1\nNorth 3\nSouth 5\nEast 1\nOptions:\nA) (-5,-9)\nB) (-2,-11)\nC) (0,-7)\nD) (-2,-10)",
  "answer": "D",
  "id": 42
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the ball.\nAlice has the key.\nGrace has the ring.\nBob has the pen.\nSwap Alice and Grace.\nSwap Bob and Grace.\nSwap Bob and Alice.\nSwap Emma and Bob.\nWho has the ring?\nOptions:\nA) Alice\nB) Emma\nC) Grace\nD) Bob",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the ball.\nAlice has the key.\nGrace has the ring.\nBob has the pen.\nSwap Alice and Grace.\nSwap Bob and Grace.\nSwap Bob and Alice.\nSwap Emma and Bob.\nWho has the ring?\nOptions:\nA) Alice\nB) Emma\nC) Grace\nD) Bob",
  "answer": "B",
  "id": 43
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nCharlie has the coin.\nBob has the pen.\nGrace has the key.\nAlice has the book.\nSwap Charlie and Alice.\nSwap Grace and Alice.\nSwap Bob and Grace.\nSwap Alice and Charlie.\nWho has the coin?\nOptions:\nA) Charlie\nB) Grace\nC) Bob\nD) Alice",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nCharlie has the coin.\nBob has the pen.\nGrace has the key.\nAlice has the book.\nSwap Charlie and Alice.\nSwap Grace and Alice.\nSwap Bob and Grace.\nSwap Alice and Charlie.\nWho has the coin?\nOptions:\nA) Charlie\nB) Grace\nC) Bob\nD) Alice",
  "answer": "C",
  "id": 44
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND False) OR (NOT True OR True)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND False) OR (NOT True OR True)\nOptions:\nA) False\nB) True",
  "answer": "B",
  "id": 45
 },
 {
  "task": "dyck_language",
  "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())(()(())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())(()(())))\nOptions:\nA) Balanced\nB) Unbalanced",
  "answer": "A",
  "id": 46
 },
 {
  "task": "boolean_expressions",
  "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (NOT True AND False)\nOptions:\nA) False\nB) True",
  "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (NOT True AND False)\nOptions:\nA) False\nB) True",
  "answer": "A",
  "id": 47
 },
 {
  "task": "navigate",
  "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,8)\nSouth 3\nNorth 4\nNorth 5\nSouth 3\nWest 1\nEast 5\nOptions:\nA) (9,8)\nB) (5,13)\nC) (6,11)\nD) (8,10)",
  "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,8)\nSouth 3\nNorth 4\nNorth 5\nSouth 3\nWest 1\nEast 5\nOptions:\nA) (9,8)\nB) (5,13)\nC) (6,11)\nD) (8,10)",
  "answer": "C",
  "id": 48
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nFrank has the key.\nAlice has the map.\nGrace has the ball.\nEmma has the coin.\nSwap Alice and Frank.\nSwap Emma and Grace.\nSwap Frank and Grace.\nSwap Alice and Emma.\nWho has the ball?\nOptions:\nA) Frank\nB) Grace\nC) Alice\nD) Emma",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nFrank has the key.\nAlice has the map.\nGrace has the ball.\nEmma has the coin.\nSwap Alice and Frank.\nSwap Emma and Grace.\nSwap Frank and Grace.\nSwap Alice and Emma.\nWho has the ball?\nOptions:\nA) Frank\nB) Grace\nC) Alice\nD) Emma",
  "answer": "C",
  "id": 49
 },
 {
  "task": "tracking_shuffled_objects",
  "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nBob has the coin.\nDavid has the pen.\nGrace has the book.\nCharlie has the ring.\nSwap Grace and David.\nSwap Grace and David.\nSwap Charlie and David.\nSwap Grace and David.\nWho has the pen?\nOptions:\nA) Bob\nB) Charlie\nC) Grace\nD) David",
  "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nBob has the coin.\nDavid has the pen.\nGrace has the book.\nCharlie has the ring.\nSwap Grace and David.\nSwap Grace and David.\nSwap Charlie and David.\nSwap Grace and David.\nWho has the pen?\nOptions:\nA) Bob\nB) Charlie\nC) Grace\nD) David",
  "answer": "B",
  "id": 50
 }
]"""
all_tasks = json.loads(raw_tasks_json)

checkpoint_path = "phase22_adapter_checkpoint.pt"
ckpt = torch.load(checkpoint_path, map_location=device)

def set_adapter_weights(weights_list):
    for p, saved in zip(trainable_params, weights_list):
        p.data.copy_(saved.to(device))

def disable_adapter():
    zeros = [torch.zeros_like(p) for p in trainable_params]
    set_adapter_weights(zeros)

print("\n" + "="*80)
print("2. EVALUATING 50 DEEP TASKS")
print("="*80)

teacher_model.eval()
student_model.eval()

teacher_correct = 0
baseline_correct = 0
recal_correct = 0

print(f"{'ID':<4} | {'Type':<25} | {'True':<4} | {'Teacher (4s)':<12} | {'Base (0s)':<10} | {'Recal (0s)':<10}")
print("-" * 80)

for t in all_tasks:
    ans = t["answer"]
    
    t_pred, _ = predict_mc(teacher_model, t["teacher_prompt"])
    
    disable_adapter()
    b_pred, _ = predict_mc(student_model, t["student_prompt"])
    
    set_adapter_weights(ckpt["adapter_weights"])
    r_pred, _ = predict_mc(student_model, t["student_prompt"])
    
    if t_pred == ans: teacher_correct += 1
    if b_pred == ans: baseline_correct += 1
    if r_pred == ans: recal_correct += 1
        
    t_mark = "✅" if t_pred == ans else "❌"
    b_mark = "✅" if b_pred == ans else "❌"
    r_mark = "✅" if r_pred == ans else "❌"
    
    print(f"{t['id']:<4} | {t['task']:<25} | {ans:<4} | {t_pred} {t_mark:<10} | {b_pred} {b_mark:<8} | {r_pred} {r_mark}")

print("\n" + "="*80)
print("3. FINAL SCORES ON 50 NEW DEEP TASKS")
print("="*80)
print(f"Teacher (4-shot)           : {teacher_correct}/50  ({teacher_correct/50:.1%})")
print(f"Student Baseline (0-shot)  : {baseline_correct}/50  ({baseline_correct/50:.1%})")
print(f"Student Recalibrated (0-s) : {recal_correct}/50  ({recal_correct/50:.1%})")


1. LOADING 50 COMPLEX TASKS & CHECKPOINT

2. EVALUATING 50 DEEP TASKS
ID   | Type                      | True | Teacher (4s) | Base (0s)  | Recal (0s)
--------------------------------------------------------------------------------
1    | dyck_language             | A    | B ❌          | B ❌        | A ✅
2    | boolean_expressions       | A    | B ❌          | B ❌        | A ✅
3    | boolean_expressions       | B    | B ✅          | B ✅        | A ❌
4    | dyck_language             | B    | B ✅          | B ✅        | A ❌
5    | tracking_shuffled_objects | D    | C ❌          | C ❌        | C ❌
6    | navigate                  | C    | B ❌          | B ❌        | B ❌
7    | tracking_shuffled_objects | A    | B ❌          | B ❌        | C ❌
8    | dyck_language             | B    | B ✅          | B ✅        | A ❌
9    | dyck_language             | B    | B ✅          | B ✅        | A ❌
10   | navigate                  | B    | B ✅          | B ✅        | C ❌
11   | navigate             